# Matrix-vector product: one warp per row

During LLM decoding, every linear layer multiplies a weight matrix by a single activation vector. That is $y = Ax$, with $A$ of shape `(M, K)` in row-major order.

The obvious mapping, one thread per row, is badly **uncoalesced**: at each step $k$, the 32 threads of a warp read `A[r, k]`, `A[r+1, k]`, ..., addresses $4K$ bytes apart. Instead, give each row to a **group of 32 threads** that read the row together, so a warp reads 32 consecutive floats per step.

## Launch configuration
- Block shape `(32, 4)`: `threadIdx.x` is the lane within a row (0..31), `threadIdx.y` picks one of 4 rows handled by the block. Grid: `ceil(M / 4)` blocks.
- Row `r = blockIdx.x * 4 + threadIdx.y`. Lane `l` sums `A[r, k] * x[k]` for `k = l, l + 32, l + 64, ...`.
- The 32 partial sums of each row are combined with a tree reduction in a `float32` shared array of shape `(4, 32)`.
- Rows with `r >= M` (in the last block) must still reach every `cuda.syncthreads()`, but must not read `A` or write `y`.

## Inputs and output
- `A`: `float32` `(M, K)`, `x`: `float32` `(K,)`, with $M, K \ge 1$.
- Return `y` as `float32` `(M,)`. Compared with relative tolerance $10^{-4}$.

## Runtime
GPU in Colab; checked here on Numba's CUDA simulator.

**Runtime → Change runtime type → GPU** before running.

Write your solution in the next cell, then run the test cell.

In [ ]:
!pip -q install "numba-cuda[cu12]"

In [ ]:
import numpy as np
from numba import cuda, float32

LANES, ROWS = 32, 4   # block = (LANES, ROWS)

def matvec(A, x):
    """A: float32 (M, K), x: float32 (K,). Returns A @ x as float32 (M,)."""
    M = A.shape[0]
    d_A, d_x = cuda.to_device(A), cuda.to_device(x)
    d_y = cuda.device_array(M, dtype=np.float32)
    matvec_kernel[(M + ROWS - 1) // ROWS, (LANES, ROWS)](d_A, d_x, d_y)
    return d_y.copy_to_host()

@cuda.jit
def matvec_kernel(A, x, y):
    lane = cuda.threadIdx.x
    r = cuda.blockIdx.x * ROWS + cuda.threadIdx.y
    buf = cuda.shared.array((ROWS, LANES), float32)
    # TODO: strided partial dot product per lane, tree reduction over the 32 lanes, lane 0 writes y[r]
    pass


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]], "dtype": "float32"}, {"__nd__": [1.0, -1.0], "dtype": "float32"}], "e": {"__nd__": [-1.0, -1.0, -1.0], "dtype": "float32"}}, {"a": [{"__nd__": [[2.0, 0.0, 1.0]], "dtype": "float32"}, {"__nd__": [3.0, 7.0, -1.0], "dtype": "float32"}], "e": {"__nd__": [5.0], "dtype": "float32"}}, {"a": [{"__nd__": [[1.8267565965652466]], "dtype": "float32"}, {"__nd__": [-3.07833194732666], "dtype": "float32"}], "e": {"__nd__": [-5.623363018035889], "dtype": "float32"}}, {"a": [{"__nd__": [[0.9580639600753784, 0.0696372240781784, 1.318250060081482, 0.38562923669815063, 1.827258586883545, 0.03174375742673874, -0.5162294507026672, 0.580484926700592, 0.4321068525314331, -0.3568393588066101, -0.2473038285970688, 0.7194406986236572, 0.7043160200119019, -0.4939342439174652, -0.36771371960639954, -1.8067903518676758, 1.679207444190979, -0.2242908775806427, 1.3372774124145508, 0.41746559739112854, 1.943962812423706, 1.5371099710464478, 0.31829842925071716, 1.4807634353637695, -0.9501235485076904, 1.2586181163787842, -1.4804235696792603, 0.34323635697364807, 1.0648764371871948, 0.22363214194774628, -0.36713749170303345, -0.8055600523948669], [-0.3428000211715698, 1.051125168800354, 0.8908394575119019, -0.2621314525604248, -1.246004343032837, 0.6739972233772278, -1.449873924255371, -0.5308548808097839, -0.7348283529281616, 0.7432652115821838, 0.23594973981380463, 0.4618547260761261, 0.2724079191684723, -0.6779301166534424, 0.5354759693145752, 1.4124603271484375, -0.03677124157547951, 0.6335944533348083, -0.1259031891822815, 1.0285553932189941, 0.6666366457939148, 0.8758194446563721, 0.3484230041503906, 1.6400039196014404, -0.3611505925655365, -0.3341660797595978, -0.5918599963188171, 0.6109626889228821, -0.6224002838134766, -0.6445326209068298, 0.7274748086929321, 1.1620957851409912], [0.5743514895439148, -2.683103561401367, -0.9536324739456177, -1.0734894275665283, -1.2075111865997314, -0.407767653465271, 1.8175432682037354, -0.30739855766296387, -0.7354457974433899, 0.6988227367401123, 0.0691717267036438, 0.12068112939596176, -0.6866741180419922, -0.0015044669853523374, 0.4345371723175049, -1.911306619644165, -1.6102243661880493, 1.084560513496399, 0.843004584312439, -0.5066567063331604, -1.020203948020935, -0.1855466216802597, -1.1936522722244263, -0.34009698033332825, 1.6511003971099854, 0.1320718377828598, 1.1662884950637817, -0.22262053191661835, 0.3694155514240265, 1.3858305215835571, -1.3359375, -0.3474780023097992], [-0.15689849853515625, 0.2819811701774597, 1.5536667108535767, 0.6855270266532898, -1.497666597366333, -0.27097204327583313, -0.8936434388160706, -0.9190470576286316, 2.024827718734741, -1.0629706382751465, 0.23639518022537231, -0.7152122259140015, 0.8609809279441833, 0.35370323061943054, 0.12742771208286285, -0.07616713643074036, 1.1377168893814087, -0.9316998720169067, -0.7269383072853088, -1.1166059970855713, 1.334546446800232, -0.29356637597084045, -1.037400722503662, -0.4190734028816223, 2.4996471405029297, -0.10747295618057251, 0.8500205874443054, 0.465181827545166, -0.01238328218460083, -0.692704975605011, -0.33429524302482605, -0.3544524013996124]], "dtype": "float32"}, {"__nd__": [-2.2092761993408203, 2.590132236480713, 0.924836277961731, 0.1162865087389946, -0.20334914326667786, -0.7896443605422974, 0.9496217370033264, -0.1868559867143631, 0.4792725443840027, -1.3874951601028442, 0.3784765303134918, 2.0003011226654053, -0.28962886333465576, 0.7999549508094788, 0.03533363714814186, 1.1051905155181885, 1.0674245357513428, 0.06164637207984924, -0.559585452079773, 0.28675416111946106, 0.04176902770996094, -0.6179032921791077, 0.4564788341522217, 0.4323228597640991, -0.8890265822410583, -0.11148789525032043, 2.9211394786834717, 0.8168345093727112, 0.09777931123971939, 2.511082410812378, -0.455403208732605, -2.476854085922241], "dtype": "float32"}], "e": {"__nd__": [-1.6736176013946533, -1.7353078126907349, -5.989441394805908, 4.261524677276611], "dtype": "float32"}}, {"a": [{"__nd__": [[-1.906941533088684, -1.5195109844207764, 0.2600550651550293, 0.9763562679290771, -2.1701903343200684, 1.117243766784668, -1.2277384996414185, 0.2564348578453064, 0.46391960978507996, 0.43232983350753784, 0.007547080051153898, 0.17099551856517792, -1.1398200988769531, 0.2782168984413147, 0.9796523451805115, 0.9262226223945618, 2.6128528118133545, 1.0832325220108032, 0.7264273762702942, 1.6635005474090576, 0.24113763868808746, -1.28987717628479, -2.2398598194122314, 1.2765709161758423, 0.568031370639801, 0.847296953201294, -0.3328748941421509, 0.34482941031455994, 1.882381558418274, 0.33483418822288513, -1.1155186891555786, -0.4314497113227844, 1.38511061668396], [-1.089961051940918, -2.5814294815063477, -2.293955087661743, -1.3389389514923096, 0.7993935346603394, 0.0884632021188736, -0.7572771310806274, -1.2851365804672241, -0.11398930847644806, -0.15959563851356506, -0.5639269351959229, -0.24525414407253265, -0.7631818056106567, 1.8032219409942627, -2.366941452026367, 0.12218885868787766, -1.014405608177185, 0.706516444683075, -0.2351061999797821, 0.36602145433425903, 0.5315962433815002, 0.20532624423503876, 0.9910404682159424, -1.3914777040481567, 0.951698899269104, -0.21894244849681854, 0.38064438104629517, 1.5565333366394043, 1.1244068145751953, 1.9839062690734863, -2.25059175491333, -0.955881655216217, 0.9641676545143127], [-1.1306781768798828, -0.18259280920028687, 1.7078758478164673, 0.8747000098228455, -0.9622686505317688, 0.45380643010139465, 0.6453005075454712, 0.06737638264894485, -0.10050634294748306, 0.1415574699640274, -1.5005136728286743, 0.12651242315769196, -0.9669110774993896, 0.39357107877731323, 0.4292169511318207, -0.8927932977676392, 0.6126711368560791, 0.3558124303817749, -0.04209628701210022, 0.725806713104248, -0.3127966821193695, -0.28415799140930176, 0.07778617739677429, -0.05922280251979828, 1.1421668529510498, -2.3739452362060547, -0.31047356128692627, -2.014955997467041, -0.5262386798858643, 0.23222823441028595, -0.0221461933106184, 0.9393318891525269, 0.2866276502609253], [-2.776874542236328, -1.158342957496643, 1.5136624574661255, 1.5159204006195068, 0.052347078919410706, -1.6017876863479614, 2.1188039779663086, -2.1530354022979736, 0.2502431571483612, -0.1118730828166008, 0.17806975543498993, -1.2407174110412598, -0.4722427725791931, 1.0021493434906006, -0.5663898587226868, -1.4614338874816895, -0.0968598946928978, 0.006268835626542568, -0.22062526643276215, -0.8773841261863708, 0.4550834596157074, -0.2132079303264618, -1.2648979425430298, 0.13264484703540802, -0.11691341549158096, 0.05893535912036896, -1.793182373046875, -0.8577885627746582, 0.3729998469352722, 0.8514968752861023, 0.26848822832107544, 0.08492853492498398, 0.6574751138687134], [0.9197622537612915, 1.7089096307754517, 0.4161616563796997, -0.6178988218307495, 0.43328857421875, -1.994313359260559, 0.020228339359164238, -0.4773692488670349, -0.0515926256775856, 0.46636250615119934, 0.028592398390173912, -1.4722402095794678, -0.376668244600296, -0.8190875053405762, -0.03521423414349556, 0.34775832295417786, -0.9762750267982483, 0.723614513874054, 1.2932844161987305, -0.2788844108581543, 0.7214635610580444, -1.4895076751708984, -0.5475362539291382, 0.957765519618988, -1.7717266082763672, -0.5123825669288635, -0.9021143913269043, 1.1438186168670654, 1.1407461166381836, -0.6291700005531311, 0.11823820322751999, -2.6185953617095947, 0.8450719714164734]], "dtype": "float32"}, {"__nd__": [0.34489306807518005, 1.32167649269104, -2.393613338470459, 0.3424515426158905, -0.35849234461784363, -2.6175827980041504, -0.6984831690788269, -0.08622463047504425, 0.7974618077278137, 0.1902010291814804, -0.5223810076713562, 0.43909767270088196, -0.11440622806549072, -0.6565077304840088, -1.0265802145004272, 0.14284253120422363, 0.15089504420757294, 1.835772156715393, -0.45719438791275024, 1.599266529083252, 0.3460620939731598, 1.173685908317566, -0.9101955890655518, -0.8784043192863464, -0.31216081976890564, 0.8927077651023865, 0.1732492744922638, 0.7507971525192261, 0.31595873832702637, 1.027280569076538, 0.19700369238853455, -0.5264894962310791, 0.5140479207038879], "dtype": "float32"}], "e": {"__nd__": [1.9689717292785645, 9.0554780960083, -8.505508422851562, -3.2938947677612305, 7.163816928863525], "dtype": "float32"}}, {"a": [{"__nd__": [[0.09003371000289917, 1.748336672782898, -0.15972213447093964, 0.10902445018291473, -0.30643951892852783, 1.1561105251312256, 0.8746158480644226], [0.6510992050170898, -1.2466926574707031, -1.516545057296753, 0.2648726999759674, 0.7220378518104553, 1.4791980981826782, -1.795693278312683], [-0.17030522227287292, -1.6904635429382324, -0.9607281684875488, -0.22110231220722198, 0.9630199074745178, -0.5266167521476746, -0.7181774973869324], [-0.31983038783073425, 1.2576661109924316, 2.4159128665924072, -1.5420101881027222, -0.2734977900981903, -0.43029892444610596, 1.1688016653060913], [0.43566620349884033, 1.540729284286499, -1.0121275186538696, 0.013772513717412949, -0.45555511116981506, -1.096416711807251, -0.15803399682044983], [-1.1844993829727173, 2.172621726989746, 0.4172911047935486, -1.0164598226547241, -0.6060484051704407, 1.7326364517211914, -0.478118360042572], [0.6941054463386536, 0.5836544036865234, 1.0927317142486572, 0.6347154378890991, -0.2101142555475235, -0.5914984345436096, -1.7757763862609863], [-0.04765275865793228, 0.13870155811309814, -0.3601946234703064, -0.14363735914230347, -1.2139527797698975, 2.171170234680176, -0.5010127425193787], [-1.035396695137024, -0.5970107913017273, -0.16847090423107147, 0.7418992519378662, -2.4273548126220703, 0.515755832195282, -0.615847647190094], [-0.17923468351364136, 1.0174627304077148, 0.7666960954666138, -0.6781798005104065, -0.0927329957485199, 1.8740146160125732, 1.363098382949829]], "dtype": "float32"}, {"__nd__": [0.8707621097564697, 0.011627346277236938, 0.9657787680625916, 0.4004381597042084, -0.5452161431312561, -0.997035801410675, 0.5270590782165527], "dtype": "float32"}], "e": {"__nd__": [-0.5365058183670044, -3.621041774749756, -1.5628598928451538, 2.6460514068603516, 0.6835496425628662, -2.659243583679199, 1.6890521049499512, -2.2121987342834473, -0.28952881693840027, -0.7748174667358398], "dtype": "float32"}}, {"a": [{"__nd__": [[0.38650283217430115, 1.2115569114685059, -0.478994220495224, 1.4068458080291748, -0.3610649108886719, 1.2603949308395386, 0.5507586598396301, -0.5727424025535583, -0.9430860280990601, -0.11643341928720474, -0.26016443967819214, 0.9442658424377441, -0.6876183152198792, -1.474273443222046, -0.014948368072509766, 1.9265888929367065, 1.6422021389007568, -0.6180402040481567, -0.27046045660972595, 0.32261648774147034, -0.6595926880836487, -0.2415473908185959, -0.271843284368515, -1.0465047359466553, -0.6149859428405762, -0.4814890921115875, 0.23836323618888855, -1.1797868013381958, 0.10856445878744125, -0.9789382219314575, -0.8058544397354126, -1.1210262775421143, 4.015010356903076, 1.2599880695343018, 1.3482909202575684, 0.8161882758140564, -0.23215779662132263, 0.07068551331758499, 0.2922717332839966, 0.45044153928756714, 0.5978001952171326, 1.5296721458435059, -2.9571878910064697, -0.5523435473442078, 0.680171549320221, -0.3554946184158325, -0.1541263461112976, 1.4936680793762207, 0.194073885679245, 1.7850669622421265, -0.041467588394880295, 0.23957011103630066, -1.064625859260559, -1.9856423139572144, -0.41670241951942444, 0.06352199614048004, -0.3439176082611084, -1.346735954284668, 0.6585364937782288, 0.07284577935934067, 0.010964881628751755, -0.7770242094993591, 1.6502982378005981, 1.479138731956482, 0.30928856134414673, -0.11463054269552231, 0.25414302945137024, 1.697879672050476, 2.811840057373047, 0.013234156183898449, -0.42579028010368347, -1.8115838766098022, 2.7992970943450928, -1.103662133216858, 0.8742792010307312, 0.5016737580299377, 1.3359389305114746, -0.8596674203872681, -1.6201274394989014, 0.15047596395015717, 0.4127308428287506, 1.1513655185699463, -1.6848421096801758, -0.4526776671409607, -0.2618641257286072, -1.3526510000228882, -0.16133297979831696, 0.8022027015686035, -0.03215410187840462, -1.4381428956985474, 0.3487955331802368, 0.8612946271896362, 0.5389732718467712, 0.7925052046775818, -1.6518425941467285, 0.4636634886264801, 0.6031314134597778, -0.37379369139671326, -0.05242816358804703, -0.7700425386428833, 1.2389658689498901, -0.2961309850215912, -0.9423303604125977, 0.18934915959835052, 0.11445123702287674, 0.6718542575836182, -0.8322688341140747, -0.674513578414917, 0.3867115080356598, 2.2409212589263916, 0.47039511799812317, -1.3610966205596924, 0.1720748245716095, -0.7765471935272217, -1.8503470420837402, -0.6822056770324707, -0.5729925632476807, 0.42174381017684937, 1.3251672983169556, 1.1429766416549683, 0.813230037689209, -0.42983052134513855, 0.15966148674488068, -0.03665835037827492, 1.4138050079345703, -0.6498426198959351, -0.5474255681037903, 1.5014197826385498, 0.581462562084198, -0.4462331235408783, 0.1689763367176056, 0.1537400335073471, 0.1932879537343979, 0.8058782815933228, 0.3717736601829529, -0.1795397251844406, -0.5024283528327942, -1.7252230644226074, -0.43279680609703064, 1.1511259078979492, 0.23226463794708252, -0.6488568782806396, -0.23847316205501556, -0.8530846238136292, 0.29613813757896423, 2.0159146785736084, 1.8162868022918701, 1.2076218128204346, 0.789933979511261, -0.21979403495788574, -1.2355077266693115, 0.019639477133750916, -0.4406792223453522, 0.8423542380332947, 0.16933010518550873, -0.5659253597259521, 0.46401605010032654, 2.9363458156585693, -0.8966197371482849, 1.1501190662384033, 0.6092477440834045, -0.08420709520578384, -0.7451052069664001, 0.03494150936603546, 0.23494111001491547, 0.7859030961990356, 0.7207475900650024, 0.33564358949661255, -0.6681472659111023, 0.31849390268325806, -0.4522289037704468, -1.030527114868164, 0.9123314619064331, -0.6508362889289856, -1.1833113431930542, 0.09823982417583466, -1.9197437763214111, -0.05340863764286041, 0.46027740836143494, -0.4099349081516266, 0.24871130287647247, 2.167280673980713, 1.4355412721633911, 0.9425172209739685, 0.60926353931427, -0.5095142126083374, -0.1322103589773178, -0.27942389249801636, 0.033601392060518265, 0.12272394448518753, -0.8049627542495728, 1.6188915967941284, -1.0382230281829834, 0.3666912019252777, 1.0378625392913818, -0.026483602821826935, -0.6680334806442261, 0.033419422805309296, 1.9428584575653076, 0.6261038780212402], [-0.2896069288253784, 0.5850505828857422, -0.19636918604373932, -1.8765511512756348, 0.2443511039018631, 1.0430858135223389, 1.1554746627807617, -0.06525295227766037, -1.412984848022461, 1.1992993354797363, -0.15496523678302765, 0.7376782298088074, 1.130954623222351, 0.3386093080043793, -0.06246346980333328, -0.6303871870040894, -0.4355660080909729, 0.02830701693892479, -0.46638649702072144, 0.3600184917449951, -1.9614372253417969, -1.0189257860183716, -0.03862045332789421, -0.7034804224967957, -0.5474863648414612, -1.1492044925689697, 0.6610785126686096, 1.6204195022583008, -1.0199918746948242, 1.1366065740585327, -1.4700734615325928, 0.12453320622444153, -2.0203676223754883, 0.46881750226020813, 0.0036097730044275522, 0.12210457772016525, 1.198567509651184, 0.5728311538696289, -0.04823875427246094, -0.8424931168556213, 0.02666337415575981, -0.7181218862533569, -0.44833970069885254, 1.4526641368865967, 0.8495866060256958, 0.2602483332157135, 0.6003328561782837, -1.4725618362426758, 0.0762244462966919, -1.7234350442886353, -0.6984401941299438, 0.3361152708530426, 0.34210526943206787, 0.6642372012138367, 0.11605605483055115, 0.008802109397947788, -0.04673934727907181, 1.3597087860107422, 1.2530686855316162, -1.5053163766860962, 0.07836554199457169, 1.2380720376968384, -0.31057482957839966, 0.16271592676639557, -0.6616724729537964, -1.2639461755752563, 0.06622907519340515, -0.48092004656791687, -0.2385380119085312, -1.529490351676941, -1.2471624612808228, -0.22655127942562103, -0.5447095632553101, 0.4433755874633789, -1.1642268896102905, 0.5917767286300659, 0.4475606381893158, -0.03319079056382179, 0.5150267481803894, -0.21642622351646423, -0.3321765065193176, -0.16966304183006287, -1.4113640785217285, -0.18801479041576385, 0.6438881754875183, -1.7600427865982056, 0.5280057787895203, -0.7939373850822449, -0.5173986554145813, -0.07322858273983002, 0.08427481353282928, 1.197696328163147, -0.30942636728286743, 1.5460668802261353, -0.1225753128528595, 0.3228130638599396, 0.5140435099601746, -0.8229801058769226, -1.271087884902954, -0.03993978351354599, 0.5501641631126404, 0.7179550528526306, -0.49595627188682556, 1.6254194974899292, -0.24865861237049103, -0.7929532527923584, 1.0531949996948242, 1.127305030822754, -1.0327833890914917, 0.7539494633674622, -1.326534390449524, -0.011146479286253452, 2.2225890159606934, 1.256201148033142, 0.6820923089981079, 0.6328900456428528, -0.8301624655723572, 0.33287736773490906, 0.4535401165485382, -1.0224860906600952, 0.21752507984638214, 0.5647282600402832, 0.23615916073322296, 0.4539389908313751, -0.19956548511981964, -1.0907214879989624, 0.06515006721019745, -1.8843028545379639, -0.6782270669937134, 1.1282567977905273, 1.822555422782898, 0.44775065779685974, -2.3638741970062256, 1.3792015314102173, 0.5875310897827148, -0.14244890213012695, 0.7963337302207947, 0.5948165655136108, -1.1406171321868896, -1.2444143295288086, 1.089666724205017, -2.237473964691162, 0.6528853178024292, 0.37071892619132996, 1.034990906715393, 1.7988975048065186, 1.3954088687896729, 1.6155163049697876, 0.8849202394485474, 0.5726985335350037, 1.1388957500457764, 0.060974378138780594, -0.8375439643859863, -0.005138245411217213, -1.5233927965164185, -0.4762767553329468, 2.1337692737579346, 0.5898863673210144, -0.104012630879879, -0.5286069512367249, -1.6987944841384888, -0.4690552055835724, -0.7093132138252258, 0.06291770190000534, 0.14164426922798157, -0.00807039812207222, -1.8494209051132202, 1.5060701370239258, 0.4952414035797119, 0.06110600382089615, -0.27262139320373535, 0.489059180021286, 1.2918702363967896, 0.6525207757949829, -1.1143360137939453, -1.2102105617523193, -2.320026159286499, 0.15964266657829285, 0.7433220744132996, 1.242090106010437, 0.15513943135738373, 0.8338120579719543, -0.8992630839347839, 0.5323162078857422, -0.11481258273124695, -0.6581799387931824, -1.6737364530563354, 1.156531572341919, -2.2498652935028076, 0.6252995729446411, -0.802219033241272, 0.605822741985321, -0.5549833178520203, 0.7501620650291443, -0.2201387733221054, -1.0477230548858643, -0.5206915736198425, -1.6154813766479492, 0.26982662081718445, 1.0361037254333496], [0.4142778813838959, 0.10155212134122849, 1.2841352224349976, -1.1704825162887573, 0.5507282614707947, 0.0051128738559782505, -0.4749050736427307, 0.5451962947845459, -1.1485017538070679, -0.13800767064094543, 0.3336654305458069, -1.168419599533081, -0.23796668648719788, 0.5225855708122253, 0.9484075903892517, -0.9456542134284973, 0.8959514498710632, -0.2982275187969208, 0.08971983194351196, -0.5518433451652527, 0.7771668434143066, -0.7082753777503967, 0.27944210171699524, 0.3755491077899933, 0.7516474723815918, -1.0428359508514404, 0.7702812552452087, 0.0294971764087677, 1.4479398727416992, -0.3476806581020355, -0.5720516443252563, -0.061392642557621, -0.31712234020233154, -0.17466117441654205, -1.2737936973571777, -2.087588310241699, -1.4747819900512695, -0.5981080532073975, 0.4211537539958954, -0.319654643535614, -0.968058705329895, -0.1642845720052719, 0.05345994979143143, 2.7524750232696533, 1.2700642347335815, 0.17909152805805206, -1.4630720615386963, -0.18444138765335083, -0.313263863325119, 0.5465628504753113, 2.1136679649353027, -0.07376551628112793, -1.9787038564682007, -0.12475525587797165, -1.6850863695144653, 0.8307342529296875, 1.0978639125823975, 0.3287368714809418, -1.4913262128829956, -0.18334804475307465, 1.208828330039978, -1.0075623989105225, -0.595819354057312, 0.2521924674510956, 0.49409934878349304, 0.07677619904279709, 0.3376830220222473, 0.2061121165752411, -0.5821773409843445, -0.08351884037256241, 0.28280600905418396, -1.0921801328659058, 0.1939898580312729, 0.0971963107585907, 0.46132692694664, 0.347756028175354, -1.0491350889205933, -1.258439064025879, 0.25008273124694824, -0.15061555802822113, 0.10054108500480652, -0.563538134098053, -1.2778362035751343, 0.2675737738609314, -0.1321062445640564, 0.6137542128562927, 1.0908143520355225, 1.0918139219284058, 0.9876492023468018, -0.9601809978485107, -2.1120848655700684, -0.6170183420181274, -1.2131603956222534, -1.323216199874878, 0.6497148275375366, -1.6165415048599243, 0.9054979681968689, 1.087734580039978, 2.6532206535339355, 1.0304796695709229, 2.088303565979004, -0.6487269401550293, -0.6892934441566467, 0.9701080918312073, -1.3215715885162354, 1.1325098276138306, -0.28140175342559814, -1.9719762802124023, 1.8421061038970947, -1.1265901327133179, 2.4894070625305176, -1.4824445247650146, -0.8530563712120056, 0.20653612911701202, 0.00630975840613246, -1.5863792896270752, 0.10066196322441101, 0.17944872379302979, 0.4475690722465515, -1.6133707761764526, -0.6118007898330688, -0.11296556890010834, 0.7982983589172363, 0.15912777185440063, -0.3530378043651581, -0.43053120374679565, 1.452958583831787, -0.8240624666213989, 0.2922510504722595, -2.1742522716522217, 1.6942808628082275, 0.4529268741607666, -1.76482093334198, 0.5994302034378052, 1.3820371627807617, 0.8122643828392029, 0.07000603526830673, 0.8404346108436584, -0.7480292320251465, -0.2588430643081665, -0.3622119724750519, 0.07093449681997299, -1.0481683015823364, 1.372219443321228, -0.12945406138896942, -1.0834840536117554, -0.2228236198425293, 0.3603581190109253, 0.04042252525687218, 1.0949475765228271, -0.3619973063468933, -0.7752352952957153, 0.4961172938346863, 1.2738149166107178, -0.9710970520973206, -1.5918551683425903, -0.6492876410484314, -0.08174177259206772, -1.0354732275009155, 0.15550249814987183, -0.8705102205276489, 0.8839381337165833, -1.195253610610962, -0.9227498769760132, 0.06125394254922867, 0.6640990376472473, 0.9097403287887573, -0.7009552717208862, 0.005886630155146122, 0.5257308483123779, 1.4118064641952515, -0.9054898023605347, -1.2608896493911743, 0.46770378947257996, -0.2671178877353668, 1.331496238708496, 0.008740151301026344, 0.3080451190471649, -0.8016861081123352, 0.26254361867904663, 1.2063270807266235, -0.26474371552467346, -1.2093778848648071, 0.8669277429580688, -0.3272186517715454, 0.03759843111038208, -0.7326170802116394, 0.9229820966720581, -0.7643222808837891, -0.5017606616020203, -0.012400643900036812, -0.2215316891670227, -2.2626779079437256, 0.4496886730194092, -1.186792016029358, 0.23659688234329224, -0.41109034419059753, -0.14485999941825867, -0.7568107843399048, -1.083967685699463], [0.1215178519487381, 0.19358517229557037, 0.9865535497665405, 0.5155532360076904, -0.7235081195831299, 1.1442936658859253, 2.1819169521331787, 0.693253219127655, -0.21459431946277618, -0.9912552237510681, 0.06385394930839539, -1.4611167907714844, 0.7682124972343445, -0.40619340538978577, -0.6010714173316956, 0.23076601326465607, -1.267837405204773, -1.767261266708374, -0.46547597646713257, 1.1878371238708496, -0.8773794770240784, 0.841112494468689, -1.8671338558197021, -0.09455468505620956, 0.48021113872528076, 1.75497305393219, -1.7903146743774414, -0.09582646936178207, -0.9374459981918335, 0.27377140522003174, -0.583678126335144, 0.07032898813486099, -0.7871711850166321, -0.5858622193336487, 0.36704233288764954, 1.4570567607879639, 0.17598600685596466, 0.08886750787496567, -0.49562785029411316, 1.5427303314208984, -1.9707074165344238, -2.021507978439331, -0.8112369179725647, 0.4069746434688568, 0.9122158288955688, 0.5340163707733154, 0.7175655364990234, 0.1304417848587036, -0.5012674331665039, -1.0909396409988403, 0.4139263331890106, -0.9783605337142944, 0.38580411672592163, 0.7065905332565308, -0.648687481880188, -0.47927507758140564, -0.5811485052108765, 0.6487062573432922, 0.12247626483440399, -0.6988786458969116, 0.23322932422161102, 1.4685178995132446, 0.24756722152233124, 0.09258831292390823, -0.4309394359588623, -0.9590957760810852, -0.028836945071816444, 0.03355853259563446, -0.3399026691913605, 1.1443601846694946, -0.2646050453186035, -0.19064830243587494, 0.16042467951774597, 0.49729830026626587, -1.8128713369369507, 0.23078995943069458, 1.2150660753250122, -1.137378454208374, 0.35712364315986633, -0.5346667170524597, -1.0136022567749023, 0.2333964705467224, -0.1293531209230423, -2.0938384532928467, -0.6124882102012634, 0.779683530330658, -0.6108347773551941, -0.7666376829147339, -0.3940533697605133, -0.46816059947013855, 0.7104989886283875, -0.8697062134742737, -1.1843339204788208, 0.40186214447021484, 0.9591366648674011, 0.6692366003990173, 0.8073505163192749, -0.6679377555847168, -1.4835524559020996, -0.7196649312973022, -0.04522299766540527, -1.7043129205703735, -0.3210396468639374, 0.5551077127456665, -0.7054114937782288, 1.9473063945770264, 0.7237274646759033, -1.2243596315383911, -1.1400084495544434, -0.7189697623252869, 0.6822137832641602, 0.44251781702041626, -0.5736066102981567, 0.14966906607151031, 0.772907555103302, 0.49959391355514526, -1.6073123216629028, -1.0937540531158447, -0.9238395690917969, -1.729121208190918, 0.831201434135437, -0.6552144289016724, -0.6666637063026428, -0.8465259671211243, 0.28037118911743164, 0.5423267483711243, 0.6492924690246582, 1.206636667251587, -0.9398483633995056, 1.5910676717758179, 2.436213731765747, -0.5414386987686157, -0.37358731031417847, 0.5808382630348206, 1.2108525037765503, -0.28188395500183105, -0.8142946362495422, 2.0458638668060303, -0.5110553503036499, -2.733222007751465, 0.5183192491531372, 0.9765257835388184, 1.6838244199752808, 1.7480032444000244, 3.0612001419067383, -0.5626429915428162, -0.9858656525611877, -1.4540878534317017, 1.6928385496139526, -0.4414542317390442, -1.6794966459274292, -2.0576424598693848, 0.11891292780637741, -0.4361839294433594, 0.3052470088005066, -0.862024188041687, -0.7415863275527954, 0.45976558327674866, -0.20840628445148468, 0.5679107308387756, -0.1355060636997223, -0.2929134964942932, 0.3522234261035919, -0.3338576555252075, 0.16878901422023773, 0.4375035762786865, -0.517559826374054, -0.1254808008670807, 0.0939362645149231, 0.6600291132926941, 0.22660307586193085, 0.23603983223438263, 0.4526379108428955, -0.3192278742790222, 0.34548914432525635, -0.7427550554275513, -1.8258566856384277, 0.3977758586406708, -1.9598420858383179, -0.16466903686523438, 1.5149199962615967, 0.7077186703681946, -1.024963140487671, 0.7204521298408508, 0.22499941289424896, -0.8434364199638367, -0.1098717525601387, -0.5424540638923645, -0.32217487692832947, 0.3195338249206543, -1.5536657571792603, 0.8300284743309021, 1.9890038967132568, -0.16686052083969116, -0.9768466949462891, -0.4141215682029724, 1.2105835676193237, -0.5079039931297302, 0.4667705297470093, 2.263789176940918], [-0.7231724262237549, 0.09705383330583572, -0.5700691938400269, -0.36967751383781433, 0.1170865073800087, -0.7418762445449829, 0.4523126184940338, 0.17620611190795898, 1.1612439155578613, 0.6839698553085327, 0.3830125331878662, 1.4962236881256104, -0.18894369900226593, 0.6344724893569946, -0.6307652592658997, 1.4336544275283813, -0.7074540257453918, 0.9859297871589661, -0.6929429173469543, 0.24380528926849365, -1.0253357887268066, -0.7271009683609009, -0.9044414758682251, 0.47686535120010376, 0.5680517554283142, 0.7994596362113953, 0.44900667667388916, -0.14832571148872375, 0.7238885164260864, 1.610827088356018, -0.30399173498153687, -0.6673916578292847, 0.6379453539848328, 2.110591173171997, 0.0024049936328083277, -0.23644722998142242, -0.9114506244659424, -0.6265468597412109, -0.8414252996444702, -0.8643729090690613, 1.1450847387313843, 1.3597193956375122, 0.20086640119552612, 0.2067112922668457, 0.9218200445175171, 0.13083302974700928, 0.2233761101961136, 0.7104780077934265, 0.07873336970806122, 1.3994570970535278, 3.3433470726013184, 0.12473997473716736, 0.4054220914840698, -0.6959803700447083, -0.3173444867134094, 0.2171364277601242, 0.5479338765144348, -0.05660434439778328, -1.0545189380645752, 0.5163818001747131, -0.39629068970680237, 0.5369794964790344, -0.08610078692436218, 0.005029276013374329, 0.055874478071928024, -1.9752676486968994, 1.363325834274292, -0.7710289359092712, 0.49171385169029236, 0.683422863483429, -0.4276729226112366, -1.2165101766586304, 0.7069383263587952, -0.8167572617530823, 0.05048482492566109, 0.5852422714233398, -0.3454992175102234, -1.5791666507720947, -0.5497377514839172, 0.6179208159446716, -0.6875657439231873, 0.9896419644355774, -1.4107714891433716, -0.846373438835144, -1.6374831199645996, -0.8460994362831116, 1.6032077074050903, 0.6579160094261169, -0.18076545000076294, -0.5411076545715332, 1.217711329460144, -0.04887506365776062, -0.9169337749481201, 1.6083085536956787, -1.2434449195861816, 0.25046828389167786, 0.7000219821929932, 1.0181289911270142, 2.0188143253326416, 0.8900254368782043, -0.6391757726669312, 1.195995569229126, 0.35024526715278625, -1.1676905155181885, -1.0283946990966797, -0.7655251026153564, -2.111659049987793, 0.46637052297592163, 0.5945751667022705, 0.5197609066963196, -0.7922329306602478, -0.02396240457892418, 1.2159638404846191, 0.40419289469718933, -0.11806651949882507, -0.7136350870132446, -0.018587974831461906, 1.2941796779632568, 0.04682822525501251, 2.306591272354126, 2.4369754791259766, 0.734538197517395, -1.600783109664917, -0.23558059334754944, 1.291766881942749, -0.5365603566169739, -0.6670299768447876, 0.04185165837407112, 2.6933677196502686, -0.4537692964076996, -1.9255388975143433, 0.3028927445411682, 0.4130943715572357, 0.7574179172515869, 0.0892002284526825, 0.35836976766586304, -0.4533739387989044, 0.9728851318359375, -0.012046006508171558, 0.5285264253616333, -0.7250410914421082, 1.4016215801239014, 0.2560257613658905, -0.417578786611557, 0.05601584166288376, -1.0895644426345825, -0.5916461944580078, 0.8337103128433228, 0.6148226857185364, 0.8461495637893677, 0.7177664041519165, -0.268921822309494, 0.11134298145771027, 1.8966385126113892, -0.10079926997423172, -1.527734398841858, -0.011885493993759155, -0.4173445701599121, -0.10999348759651184, 0.009748228825628757, -1.0405703783035278, 0.4064355492591858, -0.1833367943763733, 0.5692316293716431, -0.7196189761161804, -1.0147780179977417, -0.4598208963871002, 1.6190237998962402, -1.8392267227172852, 1.0560879707336426, -0.09007644653320312, 1.1082093715667725, -0.43494337797164917, 0.34126073122024536, 0.3171912729740143, 2.0729658603668213, -0.17337548732757568, 0.18565769493579865, -0.8161389231681824, 0.24527406692504883, -0.7978491187095642, -0.17011240124702454, 0.32510000467300415, 0.5010623335838318, -0.9618199467658997, -0.9415929317474365, 1.108639121055603, -0.5814531445503235, 0.12439379096031189, -0.3385330140590668, -1.6272461414337158, -0.16101756691932678, -0.1767140030860901, 1.7098890542984009, -1.9512141942977905, -0.5820896029472351, 0.5601062178611755, -0.5887768864631653, -2.2030770778656006, 0.5634773969650269], [0.11301595717668533, -0.9000091552734375, -1.0478037595748901, -0.7392122745513916, 0.13294117152690887, -0.23677732050418854, -0.7163217663764954, 0.27896517515182495, 0.9772326350212097, -0.663250207901001, 1.4899704456329346, -0.035980165004730225, 0.528157114982605, 0.3477920591831207, 0.4249107837677002, 0.16110846400260925, -2.3220739364624023, -1.021207571029663, 0.7946293950080872, -1.1193976402282715, 0.02737055905163288, -1.6342350244522095, -0.6981988549232483, 1.4370665550231934, 0.15244001150131226, -0.9525130391120911, -0.7268542051315308, -1.7801212072372437, 0.3558832108974457, -0.3465251922607422, 0.6166972517967224, 0.6755572557449341, -0.4097643792629242, -1.0668455362319946, -1.3211506605148315, -0.5260226130485535, -0.11231470108032227, 1.2370514869689941, -1.0799612998962402, 0.31379827857017517, -1.1745085716247559, -0.7127522230148315, 0.6289868354797363, 0.3720860779285431, -0.6709147691726685, -0.7879875302314758, 0.9521892666816711, -0.3231487274169922, -0.4494050443172455, -1.1935861110687256, 2.3457138538360596, 1.6887389421463013, 0.3824992775917053, 2.082157850265503, 0.36646926403045654, -0.4537978172302246, 0.4309796094894409, 1.30819571018219, -1.2055360078811646, 1.142181158065796, 1.0984951257705688, 1.417262077331543, 0.4132705330848694, 1.0435433387756348, -1.7792726755142212, -0.11308328062295914, 0.9729127883911133, 0.5013329386711121, 0.22494368255138397, -1.2804738283157349, -0.20711612701416016, -1.7590967416763306, -0.27783218026161194, 0.2951955199241638, -0.5500257611274719, 0.4456641972064972, -0.02577967382967472, 0.8578847646713257, -0.3791158199310303, -1.6262537240982056, -1.2840887308120728, -0.7869983315467834, -0.026459597051143646, 0.07307395339012146, -0.7598562836647034, 0.09142988920211792, -0.6729117631912231, 0.6250157356262207, -1.3393405675888062, 1.1499557495117188, 0.27494069933891296, -0.3953324854373932, -1.0692309141159058, 0.5591391324996948, -0.5808485746383667, 0.6656216382980347, 1.8508687019348145, 0.20305289328098297, 0.27762892842292786, 0.5845356583595276, -0.47953420877456665, 1.9907095432281494, -0.1443684697151184, 0.7674277424812317, 0.42349815368652344, -0.48408952355384827, 1.4619698524475098, 0.4862474799156189, -0.5275807976722717, -0.5163500308990479, -1.778374433517456, -0.9224792122840881, 0.9791021943092346, 0.9944040179252625, -0.7681300640106201, 0.7822250127792358, 1.136113166809082, 0.9392929673194885, 1.414772629737854, -1.7190144062042236, 0.3739514946937561, -0.7750612497329712, 0.8448584079742432, -0.565291166305542, 0.2718915343284607, 1.0407123565673828, -1.39875328540802, -0.48576152324676514, -0.5406025648117065, 0.11544476449489594, -1.4176619052886963, 0.9189150333404541, 2.048274517059326, -0.1538051813840866, 0.6333823800086975, -2.5327701568603516, -1.0917696952819824, 0.7542370557785034, 0.3814716935157776, 0.7481553554534912, 1.407186508178711, 0.171371728181839, -0.17000891268253326, -0.39622893929481506, 0.16784140467643738, -2.0492119789123535, 1.528622031211853, -2.1127326488494873, -1.5375896692276, 0.10520374774932861, 0.9682373404502869, 0.575650155544281, 1.5183576345443726, -0.4139949083328247, -0.623506486415863, 0.5676472187042236, -1.1050031185150146, 0.22072859108448029, 0.522184431552887, 0.3052465617656708, -2.0223019123077393, -0.22747084498405457, -2.0348386764526367, 0.8553742170333862, -0.43800088763237, 0.9893076419830322, 1.3115365505218506, -1.3475371599197388, 0.9075148701667786, 0.3933284282684326, 1.4877996444702148, -1.2619246244430542, -1.273620367050171, -0.7876749038696289, -1.8410521745681763, 0.12286150455474854, -0.7244023084640503, 0.18965278565883636, 0.33337754011154175, 0.8286624550819397, -1.3311611413955688, 0.7191262245178223, 0.7118955850601196, -1.004725694656372, -0.6327580213546753, -1.4686652421951294, 1.1996169090270996, 1.315615177154541, -0.8690211176872253, -0.13080786168575287, 0.37573978304862976, 0.6708853244781494, -0.09925352782011032, 1.6177996397018433, -0.9634950160980225, 0.8266293406486511, 0.06467942148447037, 0.2613316774368286, -0.7632781863212585, -1.7530707120895386], [-0.5468441843986511, 0.4415927827358246, -0.8607746958732605, -1.716618537902832, -0.19865894317626953, -0.05292048305273056, -1.028702735900879, -0.1465432345867157, 0.37996768951416016, 0.17681942880153656, 0.8824663162231445, 1.1482384204864502, -0.06461270898580551, -0.8348608016967773, -1.767059087753296, 0.9657735228538513, 0.3118235766887665, 0.9715597033500671, -0.8493519425392151, 0.09210927039384842, -0.8413611054420471, -0.9279946088790894, -1.4833877086639404, -0.2612299919128418, 0.43986770510673523, 0.7383082509040833, 0.20193950831890106, -0.31630104780197144, 1.968917965888977, 1.5019629001617432, 0.9787485003471375, 0.11384434252977371, 1.2301392555236816, -0.7912428975105286, -1.3030693531036377, 0.5896487236022949, 0.13553768396377563, 1.0098938941955566, -0.8163918256759644, 0.2579323947429657, -1.0703152418136597, -0.37620338797569275, 0.43027976155281067, -0.10019189864397049, -0.9706488847732544, -1.4790925979614258, -0.41844281554222107, -0.9517046213150024, -1.4696910381317139, 0.05265706777572632, -0.9404695630073547, 1.2350375652313232, 2.0330770015716553, 1.0818620920181274, 2.752122163772583, -1.2449170351028442, -1.7410012483596802, -0.1728101670742035, -1.7858127355575562, 0.11355924606323242, 0.8494930267333984, 0.7487286329269409, 0.9426158666610718, -0.23440130054950714, -0.09531242400407791, 0.21618403494358063, -0.08522100001573563, 0.8985418081283569, -0.43267858028411865, -0.07155722379684448, -0.7200949788093567, 0.9296237230300903, 0.09417017549276352, 0.10390852391719818, -0.5140706300735474, -1.7980337142944336, -0.4822388291358948, -2.1904726028442383, -0.09932276606559753, -1.261834740638733, 1.2356501817703247, -0.6708986163139343, -0.10174262523651123, -0.5266040563583374, -1.172978401184082, -1.5715320110321045, -2.5436511039733887, -0.6865460872650146, -2.064882755279541, 1.076335072517395, -1.330354928970337, -0.6464306712150574, -0.07188676297664642, -0.19745315611362457, 1.1169219017028809, -0.11215995252132416, -0.5300148725509644, -0.1622435301542282, 1.4855096340179443, 0.7327160239219666, -1.9899672269821167, -0.23498967289924622, -0.9473857283592224, -0.020927507430315018, 0.5866513848304749, -0.029792485758662224, 0.5628293752670288, 1.6385867595672607, -1.7721197605133057, -0.07820016145706177, 0.021199028939008713, 0.4136437177658081, 0.06362367421388626, 0.8861387968063354, -0.04937543347477913, 0.40480339527130127, -1.571861982345581, 0.16625536978244781, -1.2300441265106201, 0.5494053363800049, 1.7282483577728271, -0.07905030995607376, 0.4106321632862091, 0.7222981452941895, 1.3559833765029907, 0.768801212310791, 1.2178843021392822, -0.5317310690879822, -0.07661865651607513, -1.4510732889175415, 2.1998379230499268, -1.1725199222564697, 0.9152979254722595, 0.16637073457241058, 0.12496829032897949, -1.1589643955230713, -0.031032394617795944, 0.09331639111042023, 0.4173242151737213, -0.3257514536380768, 0.2501222491264343, -0.9436382055282593, 1.2132713794708252, -0.6346766948699951, 3.5988948345184326, -0.2956105172634125, -0.624888002872467, 0.20428843796253204, -0.10813352465629578, 1.1098604202270508, 1.244063138961792, -0.585770845413208, -0.6278762817382812, -0.7041975855827332, 0.5142398476600647, 1.022289514541626, 0.7094058394432068, -0.09020742774009705, 0.09084340929985046, 0.7156215906143188, 1.296935796737671, 0.02314324490725994, 1.230141282081604, -0.27136144042015076, 0.259357750415802, 1.1857556104660034, -0.7939895391464233, -1.6506966352462769, -1.0421545505523682, -0.08499765396118164, -0.9691756963729858, -0.06947799772024155, 1.4153538942337036, -0.2993451952934265, 1.3701659440994263, 0.9142482876777649, -1.0023345947265625, 1.0129270553588867, 0.9107785224914551, 1.0256211757659912, 0.26100021600723267, 0.32308927178382874, -0.49880358576774597, -0.8167302012443542, 0.0036268457770347595, 0.7639923095703125, -0.5356144309043884, 1.6818517446517944, -0.3731782138347626, 0.3950251638889313, -1.2600624561309814, 1.6222283840179443, 0.80512535572052, 1.3765612840652466, -1.1071611642837524, 1.530419945716858, 0.5725051164627075, -0.5069452524185181, -0.28882384300231934, -1.381693959236145], [-0.06499097496271133, 1.0040756464004517, -0.7374705672264099, 0.49513909220695496, -1.0822627544403076, -2.5364019870758057, -0.07052753120660782, 1.5793399810791016, 1.3015578985214233, -0.565034806728363, 1.3569674491882324, 1.1336069107055664, 0.46679577231407166, -0.2644661068916321, 1.0664005279541016, -0.01182188093662262, -0.3163752555847168, 1.1203705072402954, -0.6191259026527405, 0.7445191740989685, 1.545314908027649, -0.6190828680992126, 0.23230156302452087, 0.5879582762718201, -1.701650619506836, -0.4457305669784546, -0.507330596446991, -0.4976319372653961, -0.8317630290985107, 1.3160392045974731, -0.8737195134162903, -0.07963448017835617, 0.6166697144508362, 1.580049991607666, -1.5761690139770508, -1.2838739156723022, -1.0395601987838745, -0.2910478413105011, 0.9444125890731812, -0.12275513261556625, 0.16092249751091003, 1.541927456855774, -0.30543938279151917, 0.1861838698387146, 1.9907615184783936, 0.8012474179267883, -0.5865586400032043, -0.6244691610336304, 1.139201283454895, -1.1139960289001465, -0.856836199760437, 0.8296802043914795, 1.3650490045547485, 1.7008588314056396, -0.9467962980270386, 0.5489357113838196, -3.285169839859009, -0.8107802271842957, 2.9718384742736816, 1.6045339107513428, 0.7611344456672668, -0.8743405938148499, -1.2008424997329712, 1.1632174253463745, -0.09881898760795593, -0.7509102821350098, 0.7894387245178223, 0.7471385598182678, -0.635106086730957, -0.3735889494419098, 2.2796781063079834, -0.9129741787910461, 1.083204984664917, -0.6337116956710815, -0.7871981263160706, -0.9754806756973267, -0.05158914253115654, 0.04852529987692833, 0.809025764465332, -0.6730905175209045, 0.47825825214385986, -0.49277395009994507, -0.6633763909339905, 0.7016656994819641, 0.5173653960227966, -1.4533313512802124, 0.21645213663578033, -1.7677258253097534, -1.0109204053878784, 1.6707602739334106, 1.3657597303390503, -0.14584539830684662, 0.6444857120513916, 1.3078677654266357, 1.763980507850647, 1.7523466348648071, -0.7336893677711487, -0.020208382979035378, 0.13416379690170288, -2.400268077850342, -0.7348417043685913, 0.41042622923851013, -0.20520509779453278, -0.5946008563041687, -1.1660566329956055, -0.5725105404853821, -0.022488372400403023, -0.3139510750770569, 1.139778971672058, 0.8154771327972412, -1.2871806621551514, 0.935413658618927, 0.029044657945632935, 0.7627830505371094, -0.6575496196746826, 0.8966464400291443, -0.7136605978012085, 1.0195720195770264, -0.029828747734427452, -0.2585071921348572, -1.3179148435592651, -1.610634446144104, 0.8638087511062622, -1.1188827753067017, -1.1740583181381226, 0.3959372937679291, 0.01975967548787594, 0.5803695917129517, -1.3112907409667969, -2.1222569942474365, -0.907675564289093, 0.994873583316803, -0.035811420530080795, 0.43461766839027405, 0.5176786780357361, 0.16470591723918915, 0.1666765809059143, -0.20844659209251404, 0.21658925712108612, 0.3222002387046814, -0.6505431532859802, 0.5969361662864685, -0.0018493353854864836, -0.08371639996767044, -0.15730921924114227, 0.8785064220428467, -0.14439719915390015, 1.1890110969543457, -1.0004775524139404, 0.5745886564254761, -1.53922700881958, 0.052565086632966995, -0.5011278986930847, -0.4562002718448639, -1.0339288711547852, -0.0259527787566185, 0.5409347414970398, -1.8305275440216064, -0.13030052185058594, 0.18041835725307465, 0.6039357781410217, 0.3523498773574829, -1.0781608819961548, 0.9867830872535706, -0.8685605525970459, 0.48879310488700867, 0.18667560815811157, -0.5836264491081238, -0.9093582034111023, -0.21926407516002655, 1.422776460647583, 0.6303192377090454, -0.5812960267066956, 0.7173916101455688, 1.6504762172698975, 2.1431210041046143, -1.7132384777069092, 0.7185304760932922, 0.823879063129425, -0.8061841726303101, -1.8597822189331055, 1.255383014678955, -0.46310415863990784, -1.6140966415405273, -1.0675679445266724, 0.7273188233375549, 2.1745762825012207, 0.9268898963928223, -0.8571982979774475, 1.2645622491836548, -1.8664213418960571, 0.991232693195343, 0.9234642386436462, 1.7908982038497925, 0.9805452227592468, 1.1414623260498047, 0.12191624939441681, -0.3240090012550354, 0.04354578256607056, 1.93315851688385], [-0.550883412361145, -0.037222567945718765, -0.4416966438293457, 0.6917722225189209, 0.5008761882781982, 2.0546228885650635, 2.1952993869781494, 1.156290054321289, 0.9191031455993652, -0.10520371049642563, -1.4597537517547607, 0.19501107931137085, 0.07118874788284302, -1.1001379489898682, 0.24863523244857788, 1.0036053657531738, -0.8585929870605469, 0.9148151874542236, 0.4423814117908478, -0.5117728114128113, 0.14303277432918549, 0.7860252857208252, -0.1498887687921524, -0.030315907672047615, 1.350305199623108, 0.5907137393951416, 0.569054126739502, -0.4718348979949951, 0.7372592687606812, 1.5341087579727173, -0.9118852615356445, -1.6219547986984253, -0.7734887599945068, -0.10126695036888123, -0.9776982665061951, -1.6687580347061157, 1.6795696020126343, -0.393322616815567, -0.7923671007156372, -2.8604817390441895, 0.936558723449707, -1.5346144437789917, 1.3454325199127197, 0.591635525226593, 0.24091248214244843, -0.19254951179027557, 0.9868427515029907, -0.5479680895805359, 0.39543041586875916, -0.4786953032016754, -0.29399123787879944, 1.6032747030258179, 1.1433967351913452, -0.9178977608680725, -0.8471530079841614, 0.06149466708302498, -0.1118304431438446, 0.5185171365737915, -0.3615606725215912, -1.0309433937072754, 0.851961612701416, -0.20432867109775543, 0.7114765644073486, -0.8878845572471619, 1.5430611371994019, 0.29891982674598694, 0.28276005387306213, 0.40333032608032227, -1.5712931156158447, 0.7296010851860046, -0.5055705308914185, -1.1180262565612793, 0.40833428502082825, 1.7598012685775757, -0.6914130449295044, 1.583095908164978, -2.219372034072876, 0.32272377610206604, -0.586661159992218, 0.2713642418384552, 2.183260440826416, 0.7003292441368103, -1.4934561252593994, 1.4965006113052368, -1.0562076568603516, -0.5603970885276794, -0.9016335010528564, 0.14851228892803192, -0.09834994375705719, 1.6732375621795654, 0.3683803081512451, 0.7229626774787903, -0.1361512988805771, 1.16121506690979, -1.0349698066711426, -1.4425190687179565, 0.9271638989448547, 0.7888785600662231, 0.07937249541282654, 2.3122246265411377, -0.8768450617790222, 1.4769903421401978, 1.5820516347885132, -1.222352147102356, 0.5989021062850952, 0.8334861397743225, -0.0976409912109375, -0.08775646239519119, -0.014554084278643131, -0.8357809782028198, -1.355550765991211, 0.836452305316925, 0.2721535861492157, 0.07723639905452728, 0.579251766204834, -0.7757176160812378, -0.6126006841659546, 0.8605212569236755, 0.04074988141655922, 0.5374335050582886, 1.1388882398605347, 1.809524655342102, -0.8793338537216187, -0.5395975112915039, -1.4108197689056396, -0.6620339155197144, 0.5528425574302673, -0.5059802532196045, 2.2444210052490234, -0.3978612720966339, -1.7708632946014404, -0.8718477487564087, 2.1454694271087646, -0.8689576387405396, -0.03906780481338501, -0.7955455183982849, -1.0794081687927246, 1.2072350978851318, -1.172958493232727, -1.7807588577270508, -0.7218365669250488, -0.6380513310432434, 0.4752994179725647, 0.1462768316268921, 1.283409833908081, -0.17837683856487274, 0.47201213240623474, 0.5365804433822632, 1.066602349281311, -1.201523780822754, -0.3040740489959717, -2.727889060974121, -0.7730317115783691, -1.202517032623291, -0.8539469242095947, 1.5942704677581787, 1.0622845888137817, -1.500462532043457, 1.8387550115585327, 1.3147168159484863, 0.20845629274845123, -0.08046787232160568, -0.11641066521406174, -0.01629859209060669, 0.6772767901420593, -0.25819215178489685, 1.0194425582885742, -0.318491131067276, -0.683420717716217, -0.13177984952926636, 0.3409929573535919, 0.431687593460083, -0.01008804701268673, -1.5234442949295044, -1.3603088855743408, -1.726844310760498, -0.27782195806503296, 0.6866108775138855, 0.029187176376581192, -0.04730188474059105, -0.4849826991558075, -0.3437245786190033, -0.6742150783538818, 1.0719468593597412, -1.2458324432373047, -0.8883633017539978, 0.4877975285053253, -0.36500245332717896, 0.6668940186500549, -0.4613325893878937, 0.433394193649292, 0.23466132581233978, -1.2864577770233154, 0.567180335521698, 1.2767521142959595, -0.6633318662643433, -0.12437562644481659, 0.6506073474884033, 1.556775689125061, 0.37440454959869385], [2.3541128635406494, -0.38598835468292236, 1.9523229598999023, -0.7258800864219666, -0.002690602093935013, -0.8916632533073425, 1.0145825147628784, 0.659573495388031, -1.7216839790344238, -2.6078238487243652, 2.1841535568237305, -0.27108293771743774, -1.6511963605880737, -0.8813006281852722, -0.44433027505874634, 1.6046336889266968, 0.4644690752029419, 0.06601184606552124, -0.9317823648452759, 0.6883710026741028, -1.7415903806686401, 1.144287109375, 0.9930378794670105, -0.453390896320343, -1.4888906478881836, -0.6049903035163879, -0.10647507011890411, 2.027841091156006, 1.7215056419372559, -0.6296871304512024, -0.7486513257026672, -1.356175184249878, 0.3941717743873596, -0.5316721796989441, 0.7191656827926636, 0.08262159675359726, -0.5294016599655151, -0.9907830357551575, -1.5858862400054932, 0.5984381437301636, -1.3104066848754883, 0.7201768159866333, -1.64700448513031, -2.5358524322509766, 0.02395775355398655, -0.5860015749931335, 0.9295604228973389, 0.010813710279762745, -0.06349556148052216, 0.8780414462089539, -0.6453883051872253, 0.6536540389060974, 2.472222089767456, 0.24479377269744873, -0.9034851789474487, -0.015531892888247967, -0.7818406820297241, -0.31370842456817627, 1.6924299001693726, -0.9609875679016113, -1.204599142074585, 0.5447084903717041, -0.37794801592826843, -0.13998286426067352, -0.594426155090332, 0.2733118534088135, 1.50230872631073, 0.670592188835144, -0.38513582944869995, -0.31957197189331055, -0.875079333782196, -0.847316324710846, 0.9746069312095642, 0.5749966502189636, 0.5024157166481018, 0.5392088890075684, -0.5846816301345825, 0.031657684594392776, 0.5614797472953796, 1.6452656984329224, 0.30445927381515503, 0.11982614547014236, -0.7380986213684082, 1.0774646997451782, -0.19779923558235168, 0.30563023686408997, 0.44599154591560364, -0.7206485867500305, 0.3464657962322235, 1.1809710264205933, -0.8222395777702332, -0.45830804109573364, -0.7547037601470947, 0.15880165994167328, 2.1413846015930176, 0.3843885362148285, 1.2962630987167358, 0.3023602366447449, -0.3575131595134735, 0.07538828998804092, 0.29555466771125793, -0.12986090779304504, -0.003353912616148591, 0.14980366826057434, -0.5631812214851379, 1.5751917362213135, 0.49498364329338074, -0.058811865746974945, -0.5065656900405884, 0.4309263229370117, -0.7066366076469421, -1.1770740747451782, -0.3090123236179352, 1.2964657545089722, -0.1418582797050476, 0.23225247859954834, -0.8956320881843567, 0.5661347508430481, -0.4865369498729706, -0.511075496673584, -0.7154276967048645, -0.07986099272966385, -0.9928259253501892, -0.3538915514945984, -0.1436949223279953, -0.9037287831306458, -1.8856627941131592, 0.47778812050819397, -1.1913214921951294, -1.892270803451538, 0.04745529964566231, -0.006603903137147427, 1.13968026638031, 1.3104697465896606, -2.1441762447357178, -0.5284736752510071, 0.3066878318786621, -1.2943168878555298, 1.1669408082962036, 0.37954047322273254, -0.6293964385986328, -0.4834604859352112, 0.3515823483467102, 2.129608392715454, 1.9432408809661865, -1.3812071084976196, 0.6383273005485535, -1.4173336029052734, -0.09218928217887878, -0.9433489441871643, -0.2484969198703766, -0.396664559841156, 0.3988482356071472, -0.706125795841217, 0.7280862331390381, -0.08103047311306, -0.8706178069114685, 1.47853684425354, 0.3652712404727936, -0.1714733988046646, 0.12621116638183594, 0.7336657643318176, -0.09621353447437286, -0.5619644522666931, -0.11492392420768738, -0.5088387727737427, -0.49661770462989807, 1.632995843887329, 0.8093201518058777, 1.3539255857467651, -0.4667532742023468, -1.7064491510391235, -0.3444546163082123, 0.5519288182258606, -0.45559433102607727, 0.21698547899723053, -0.913348376750946, 0.5274308323860168, -0.394631952047348, 0.037970032542943954, 0.6969019770622253, -0.6944564580917358, -1.0224967002868652, -0.7737802267074585, 0.6035006642341614, -0.299558162689209, 1.0796200037002563, 0.5548640489578247, 1.1254547834396362, 0.20874322950839996, 1.2175579071044922, 0.7573316097259521, -1.876133680343628, 1.440887451171875, 0.8938486576080322, -0.3660537898540497, -1.155191421508789, -1.798447847366333, 0.18354737758636475, -0.8066619038581848], [-1.1091262102127075, -0.9357006549835205, -1.576414942741394, 0.9762282371520996, -0.8237019181251526, 0.02091001532971859, 0.19762404263019562, 0.4081558585166931, -0.11663070321083069, 0.8591071963310242, -0.5541098713874817, 2.1532175540924072, -1.532599925994873, 0.7313835024833679, 2.215183973312378, -0.9130692481994629, 1.4549368619918823, 2.0712265968322754, -0.4084274470806122, 1.5671172142028809, -0.5218011736869812, 0.3462474048137665, -0.1757536232471466, -0.9212508797645569, -1.043845534324646, -0.526829719543457, -1.4456478357315063, 0.2231065183877945, 0.17478282749652863, 1.141573429107666, 0.456428587436676, -0.11709321290254593, -0.27314916253089905, 2.4991750717163086, -1.4960846900939941, -1.2774426937103271, 2.282726287841797, 1.1189085245132446, 0.23594892024993896, 0.5894771814346313, 1.0485419034957886, -1.4076787233352661, -0.25485068559646606, 0.7726362347602844, 0.7224931716918945, 1.1691408157348633, 0.3078297972679138, -0.28784725069999695, -0.5318975448608398, 0.6907681822776794, 0.05628291144967079, 0.5510700941085815, 1.478314757347107, 1.3145594596862793, -1.1726621389389038, 1.2443701028823853, -0.8766912817955017, -1.543776273727417, 2.200939416885376, 0.37940287590026855, -0.49714869260787964, -0.31552183628082275, 1.0073809623718262, 2.1104602813720703, 1.4230893850326538, -0.11112675070762634, -1.0409939289093018, -0.001203016610816121, 1.129710078239441, 0.6202628016471863, -1.1969120502471924, 0.24144165217876434, 0.7318564653396606, -0.805108368396759, 1.0019632577896118, 1.582713007926941, 1.0353236198425293, -0.7265402674674988, -0.1536531150341034, 0.05182371661067009, -0.22529321908950806, 0.982387363910675, 0.21111813187599182, 0.1246730163693428, 0.44574475288391113, -0.31912723183631897, -1.2119642496109009, 0.2347366362810135, -0.21435193717479706, 0.7813396453857422, 0.34579339623451233, -0.14367328584194183, 1.1928367614746094, 0.09818992763757706, -1.1575629711151123, 1.6559675931930542, 1.2634918689727783, 0.08761391043663025, -1.4510771036148071, -0.8860579133033752, 1.0716596841812134, 0.028137633576989174, -0.2434956580400467, 0.5563176274299622, -0.4711095094680786, -0.39861080050468445, 1.3055106401443481, 0.6234378814697266, 0.8665753602981567, 0.700188398361206, 1.1519279479980469, 0.9405567646026611, -0.6735557913780212, -0.07582785934209824, 1.7491793632507324, 0.2932051718235016, 1.834443211555481, -0.18177743256092072, -1.2512664794921875, 0.21072666347026825, -0.3754580020904541, -0.5602225065231323, -0.6167373657226562, -1.8843475580215454, 0.6530359983444214, -0.03136279061436653, -0.7967686653137207, -1.4885914325714111, 0.7634350657463074, 0.13435785472393036, 0.7781624794006348, 0.48215818405151367, 0.8519366979598999, 0.07404649257659912, -1.3356850147247314, 0.26682692766189575, 0.9855099320411682, -1.296012043952942, -0.3413166105747223, 1.5221645832061768, -1.1231878995895386, 1.4199503660202026, -0.8311486840248108, -1.5307753086090088, -0.43582668900489807, -0.5390875935554504, 0.9837601780891418, -2.0158393383026123, -0.8933808207511902, -0.8928428888320923, -1.040183186531067, 0.0982741042971611, -0.5509833097457886, -0.02419998124241829, -0.2781130373477936, 1.0353460311889648, -0.6710696816444397, -0.005604251753538847, 0.8030185699462891, 0.47562986612319946, -0.014380965381860733, 0.7883139848709106, 0.40568554401397705, 0.43593260645866394, 0.08011019229888916, 0.9818692207336426, 1.0752722024917603, -0.5457916259765625, 0.5916450619697571, 0.4759906232357025, 0.7301297783851624, -0.641105055809021, -1.102533221244812, -0.12449263036251068, 1.1538383960723877, -0.9087536334991455, 1.0525891780853271, 0.11746714264154434, 0.2706475853919983, 0.28484269976615906, -0.24141748249530792, -0.13271914422512054, 0.7197964191436768, 1.6055978536605835, 0.2839817404747009, 0.5303655862808228, -0.7180814743041992, -0.021144304424524307, 1.146529197692871, 1.1907480955123901, -0.1656370609998703, 0.05249098688364029, -0.32167211174964905, -0.738832950592041, 0.5961370468139648, -0.3180397152900696, 1.5743436813354492, -2.0556983947753906, -1.6951607465744019, -1.2028557062149048], [0.07662172615528107, -0.146636500954628, -0.7620878219604492, -3.1222455501556396, 0.1841154843568802, -1.597212553024292, 1.3810186386108398, 0.13915307819843292, -0.7245992422103882, -0.2580240070819855, 1.5833004713058472, -0.3421348035335541, -1.126213550567627, -2.541135311126709, -0.17468638718128204, -0.7649868130683899, 1.1228318214416504, 0.13979274034500122, 1.0573475360870361, -0.42901015281677246, -0.8447040319442749, 0.9338836073875427, -0.16577909886837006, -0.7278337478637695, -0.10085957497358322, 0.9959483742713928, -0.6334193348884583, 1.9286868572235107, 0.5039256811141968, -1.1833857297897339, -1.0785667896270752, -0.05851342901587486, -0.33880558609962463, -0.3716844618320465, -0.1375434547662735, 0.2619905471801758, -0.2468193918466568, -0.527576744556427, -1.8690606355667114, 1.6232225894927979, 0.12453728914260864, 0.5595053434371948, 0.1810324490070343, 1.2648844718933105, 0.03163999691605568, 1.235419750213623, -0.5535674691200256, -0.2271808385848999, 0.5351879000663757, 0.5886643528938293, 1.1869899034500122, -0.5632916688919067, 0.4661467969417572, -0.5125950574874878, -1.4004507064819336, 0.9766579866409302, -0.06664743274450302, -0.9703311920166016, -0.7462954521179199, 0.5181816816329956, -0.0014444690896198153, -0.11370739340782166, 0.6491490602493286, -0.03760072961449623, -1.7823238372802734, 0.8162660002708435, -0.48441991209983826, -0.37375837564468384, -2.385294198989868, -3.2273173332214355, 0.02751808799803257, -0.4780197739601135, -0.020401762798428535, 0.2589188516139984, 0.8656225204467773, 1.2980546951293945, 1.1890771389007568, -0.989672839641571, -0.7857160568237305, 0.10865115374326706, 1.3904250860214233, 1.107441782951355, -0.5129866003990173, -1.1783367395401, 0.28601107001304626, -0.1834953874349594, 0.6715542078018188, 0.7930706143379211, 0.3298071026802063, -0.8641093373298645, -1.1875593662261963, -0.26251354813575745, 0.4852810502052307, -0.051355645060539246, 0.7085462212562561, -0.38545748591423035, 1.0957671403884888, 0.35975632071495056, -0.2941959798336029, -0.6967887878417969, -0.11443357914686203, 0.10572060942649841, 1.579756736755371, 0.11814261972904205, 0.6852779388427734, -2.000821113586426, 0.9996795654296875, -0.04589291289448738, -0.8770105838775635, -2.208137273788452, -0.8078765869140625, 1.6290111541748047, 0.8458405137062073, 0.8128373026847839, -0.44599950313568115, 0.6332911849021912, 0.6331822276115417, 0.3564397394657135, -0.892342209815979, -0.5762864947319031, -0.28289178013801575, 3.011108636856079, -1.9321037530899048, 1.6460450887680054, 1.3967775106430054, 0.236491858959198, 0.8253347277641296, 0.3794839084148407, 0.5690209269523621, -0.563953697681427, -1.9325520992279053, 2.6811554431915283, 1.7503963708877563, -1.369516372680664, 1.1405307054519653, -0.3857475817203522, 0.4741288125514984, -1.0114266872406006, -1.043641209602356, 1.8648779392242432, -0.4447840750217438, -0.07559643685817719, 0.7080802917480469, 1.4803683757781982, -0.46838152408599854, 0.6247944235801697, 0.7703527808189392, -0.57734215259552, -0.5617460012435913, 0.6217780709266663, -0.10497324913740158, 1.5759494304656982, -1.7711920738220215, 0.5303853750228882, -0.13811102509498596, -1.3695333003997803, -0.09742662310600281, -1.4032105207443237, -0.3211364448070526, 1.094102144241333, -0.6879416108131409, 0.6158252954483032, 0.3094520568847656, -0.47789737582206726, -0.2638340890407562, 0.2240997552871704, 0.33311566710472107, -0.48600471019744873, -0.14191566407680511, 0.7813335061073303, -0.9752448201179504, -1.1553008556365967, 0.32638242840766907, 0.47686344385147095, -2.2728540897369385, 0.12018007040023804, 1.3902944326400757, 0.8392533659934998, -0.3396320343017578, 0.3919253349304199, -1.6062719821929932, 0.64543217420578, 0.3116759955883026, 0.8775521516799927, -0.13265734910964966, 0.6806203722953796, -0.411650151014328, 1.0926340818405151, 0.47353801131248474, -1.7504557371139526, -1.6173672676086426, 0.5539897084236145, 0.9953787922859192, 1.2866928577423096, -0.4789280295372009, 0.23771756887435913, -0.27767080068588257, -0.6226606369018555, -0.20658916234970093, 1.2598106861114502], [-0.4114130735397339, 1.8119463920593262, -1.2566196918487549, -0.2879292070865631, 0.5663073658943176, 0.5569887757301331, 0.17345650494098663, -1.6111243963241577, -0.22246143221855164, 1.4503120183944702, 0.7000195384025574, 0.5694031119346619, 0.4766360819339752, -0.04239661246538162, 0.012685083784162998, -0.5275203585624695, 1.4056950807571411, 0.4866303503513336, 0.4326239824295044, 0.20698727667331696, -0.19562777876853943, -1.3914309740066528, 0.02237658016383648, -0.6336767673492432, -0.5144887566566467, 0.23699544370174408, 1.3470268249511719, -0.9787402153015137, 0.04928493872284889, 1.631370186805725, -1.193405270576477, 0.5677766799926758, -0.4007743000984192, 0.7799684405326843, 1.616447925567627, -0.024477552622556686, 1.0238162279129028, 0.3621923327445984, 1.9488199949264526, -3.3399746417999268, -0.8455861210823059, -0.05638084560632706, 0.1589854210615158, 1.7204774618148804, 1.1853101253509521, -0.3112499713897705, -1.3700900077819824, 1.2188414335250854, 0.3303041160106659, -0.20382626354694366, 0.9202942848205566, 1.848158836364746, 0.41692787408828735, 1.4871234893798828, -1.7202353477478027, 0.9320449233055115, 2.414832592010498, 1.2954665422439575, -0.5093578696250916, -1.0406347513198853, 0.45720425248146057, -0.7671383619308472, -0.44213104248046875, -1.6665103435516357, -0.9215645790100098, 0.6254644393920898, -0.7052180767059326, -0.7772582769393921, -1.2769404649734497, -0.10492055863142014, 0.3873146176338196, 0.13723616302013397, -2.7970821857452393, 0.027563273906707764, -0.9207392334938049, -0.08566804975271225, -1.0794901847839355, 1.1453388929367065, 1.9179109334945679, 0.9326950311660767, 1.761918306350708, 0.26127946376800537, -0.24864281713962555, -2.0101752281188965, -0.6207100749015808, -0.4208184480667114, 1.5834262371063232, -0.9288801550865173, 0.09875825047492981, 0.6658575534820557, 0.0859876275062561, 0.011546040885150433, 1.0809513330459595, -0.8358385562896729, 0.05659894645214081, 0.2591283619403839, 1.6306430101394653, 1.4024982452392578, 0.6008769869804382, 0.8878783583641052, -2.0648579597473145, 0.33045223355293274, -0.5000439882278442, -1.500081181526184, 0.6831008195877075, -0.3887232542037964, -0.6708144545555115, -1.7225342988967896, 0.1714450716972351, 1.2317801713943481, 0.16445274651050568, 0.7347275018692017, 0.9919075965881348, -0.01724531501531601, -0.3793828785419464, -0.6185859441757202, -0.3025955557823181, -0.5542683005332947, 0.05989008769392967, -0.49325141310691833, 1.0890051126480103, -0.8860631585121155, 1.5911555290222168, -0.4705512821674347, -0.4747038781642914, 0.6590140461921692, -0.7367799878120422, -1.8841816186904907, 0.8209676146507263, 1.4160737991333008, -1.938378095626831, 0.5093221068382263, -0.569536566734314, -0.23229719698429108, 1.5544512271881104, -1.369061827659607, -0.5317641496658325, 1.2354071140289307, 0.22484079003334045, 0.4413471221923828, -0.6742023229598999, -0.8112847208976746, -0.785825252532959, 0.20555956661701202, -0.4537810981273651, 0.40907350182533264, 2.59980845451355, -1.7888997793197632, 0.2422710359096527, -0.1180923730134964, -1.0840787887573242, 0.15928182005882263, -0.9156166315078735, -1.3819080591201782, -0.028554799035191536, 1.5379384756088257, -0.24582508206367493, 0.3851791024208069, -0.6992952823638916, -0.7817243933677673, -1.4972554445266724, -0.059409212321043015, 0.4250675439834595, -1.777699589729309, -1.9579143524169922, 0.4531559646129608, -0.7447763085365295, -1.6518431901931763, 0.5826365351676941, -0.2951030433177948, -0.4867437481880188, -0.09370922297239304, -0.7336824536323547, 2.745920419692993, -2.226959705352783, -0.5866004228591919, 0.48793405294418335, 0.3137880265712738, -1.2662593126296997, -0.4110107123851776, -0.16999858617782593, 1.290571689605713, 1.556011438369751, 1.5849684476852417, 0.13838472962379456, 1.3345582485198975, 0.5468093752861023, -0.2463541477918625, -0.32002758979797363, 0.6958820819854736, 0.3322922885417938, -0.5266872644424438, -0.9256709218025208, -0.9486210942268372, 1.405737042427063, -0.23992471396923065, 0.13063748180866241, 1.4843467473983765, 1.5558395385742188, 0.8981403112411499]], "dtype": "float32"}, {"__nd__": [0.4587079584598541, -0.42972531914711, 0.07748709619045258, 0.2704310119152069, 0.7145557403564453, -0.5819495916366577, 0.8303293585777283, 1.0462090969085693, 0.3853539824485779, -1.3571178913116455, -1.8281184434890747, 0.7008045315742493, -1.7133018970489502, 0.20744507014751434, 2.8444812297821045, -0.11855773627758026, -1.33207368850708, 0.8889098167419434, -0.435440331697464, -0.07847224175930023, -1.7012027502059937, -0.6408867835998535, 1.3614927530288696, -0.26268109679222107, -0.7053789496421814, -0.04842748865485191, 0.17742416262626648, 0.9426542520523071, 0.6518017649650574, 0.9328856468200684, -1.2943865060806274, -0.36182886362075806, -1.943619728088379, 1.4896384477615356, -0.6219466924667358, -0.40396174788475037, 1.7685517072677612, 0.8425514698028564, -0.33773884177207947, -0.2761881351470947, -0.4343853294849396, 0.8248178958892822, 2.9658212661743164, -1.4055925607681274, 1.4303765296936035, 1.2244794368743896, -0.4800889194011688, 0.3037025034427643, 0.2598588168621063, -0.7201319336891174, -0.6195021867752075, 1.4418379068374634, 0.4616117477416992, -0.37267744541168213, -0.8581954836845398, -0.0770857036113739, 0.07321120798587799, 0.8537617921829224, 1.5761383771896362, 0.07398912310600281, -0.8560740947723389, 0.4106401801109314, 0.1971994936466217, -0.5124701857566833, 0.7550622820854187, -0.051956869661808014, 0.38026028871536255, 0.04067220538854599, 0.49934640526771545, 0.6330521702766418, -0.9322757720947266, 0.9478749632835388, 0.0716804713010788, -1.6132972240447998, -0.28422942757606506, 1.2745307683944702, 1.4768208265304565, -0.5716171860694885, 0.3365396559238434, -1.2202868461608887, 0.5172891020774841, -2.1656277179718018, -1.6408379077911377, 0.4639829695224762, 0.5447462201118469, -0.5931980013847351, 0.5376080274581909, 1.6074397563934326, -0.09301795065402985, 0.6407800912857056, -0.7543603181838989, -1.253487467765808, 1.8142074346542358, 0.1887386292219162, -0.5805987119674683, -1.6722253561019897, 0.6504896879196167, 0.9255449175834656, 1.434251070022583, -0.1685163676738739, 0.9599881172180176, 0.08707691729068756, -0.8555300831794739, -0.002667626366019249, -0.5425073504447937, -0.8488016128540039, -0.5903496742248535, 0.9022741913795471, 0.09364020079374313, -0.7060945630073547, -0.11410555243492126, -0.11042560636997223, -0.0818646103143692, 0.2158806025981903, -0.7060686349868774, 1.4481055736541748, -1.0889886617660522, -0.5501996278762817, -1.3312499523162842, -0.3530964255332947, -0.4890892803668976, 0.25881659984588623, -1.425775408744812, 0.26799502968788147, -1.6199705600738525, 1.3016432523727417, -0.4848363697528839, 1.2718786001205444, 1.5822222232818604, -1.08576238155365, 2.0808932781219482, 0.10946901142597198, -2.155057430267334, 0.29593396186828613, 0.2622908353805542, 1.79541015625, -1.84953773021698, -3.044786214828491, -0.06674325466156006, -0.48611366748809814, -0.9601022601127625, -0.0019695337396115065, 1.804583191871643, -0.0393218994140625, 0.9361198544502258, -0.7203637957572937, -1.3351048231124878, -0.4554721713066101, 1.8759253025054932, 1.3790500164031982, 0.8107775449752808, -0.31521645188331604, 0.38181591033935547, 0.6397772431373596, -1.5155906677246094, -0.2744789719581604, 1.9721800088882446, -1.3584022521972656, 0.24767793715000153, -0.4792095422744751, -0.3352903127670288, 1.0020923614501953, -0.8031667470932007, 0.7993660569190979, -0.022409742698073387, 1.1291911602020264, -0.8507716059684753, 1.2112410068511963, -0.008794370107352734, -0.5353925824165344, -0.7362657189369202, -2.0344650745391846, 0.9948235154151917, -0.7834998369216919, 1.2209917306900024, 2.367502212524414, -1.6574941873550415, 0.2831256687641144, -0.7382742762565613, 0.08700570464134216, 0.4373442828655243, -1.29562509059906, 1.0287195444107056, -0.210819810628891, 1.641721487045288, 0.0011391419684514403, 0.18426434695720673, 0.01449121069163084, 2.857189416885376, -0.49896106123924255, -2.4209437370300293, 0.9694898724555969, 0.08414993435144424, 0.8710277080535889, 1.2083603143692017, 1.5448030233383179, 0.2502979636192322, 0.46934473514556885, 1.1432973146438599, 1.4255105257034302], "dtype": "float32"}], "e": {"__nd__": [-0.8722996711730957, 19.998748779296875, 9.330415725708008, 5.844244956970215, 17.63341522216797, -27.575885772705078, 9.95093059539795, 30.78054428100586, 15.115235328674316, 17.967960357666016, 20.14400863647461, 4.604636192321777, -26.322784423828125], "dtype": "float32"}}, {"a": [{"__nd__": [[-1.3913297653198242, 1.9412189722061157, -1.1249171495437622, 0.23306843638420105, 1.1932936906814575, -0.50236576795578, -0.5102028846740723, 1.5570075511932373, 1.5249093770980835, -1.4352879524230957, -0.042129356414079666, 0.41578951478004456, 1.491700530052185, 0.4436294436454773, 0.9671694040298462, 0.47837838530540466, 0.839093804359436, 2.699920892715454, 0.7026450634002686, 0.5371375679969788, -1.4853466749191284, -1.0223852396011353, -1.380225658416748, -0.027858933433890343, -0.30637744069099426, -0.7328572869300842, 0.7126734256744385, -1.5537550449371338, 0.2064899057149887, -0.16118159890174866, -0.8273063898086548, -0.9070391654968262, -0.5007280111312866, 0.8029313087463379, 0.9428603649139404, -1.5353673696517944, 0.06861548125743866, -0.6893009543418884, 0.9700625538825989, 1.249950885772705, 0.8972249627113342, 0.5533168315887451, 0.7020283341407776, -1.7658389806747437, 0.4738142490386963, 0.3226320445537567, -0.16137893497943878, -0.7304857969284058, -1.2131046056747437, -0.8251623511314392, -0.6091238260269165, -2.650817394256592, -0.09608355164527893, 0.22428765892982483, 1.6288514137268066, 1.6613738536834717, 0.5181747078895569, 0.10168439894914627, 0.06068892031908035, 0.9243081212043762, 1.0894582271575928, -1.540497899055481, 0.8203302025794983, -0.443296879529953, 0.026012416929006577, -0.32769861817359924, 0.4076686501502991, -0.9879329800605774, 0.07042787969112396, -1.4709097146987915, 1.0538818836212158, 0.8822095394134521, -0.3523580729961395, -0.07620307058095932, -0.59536212682724, 0.7409249544143677, 0.5893867015838623, -0.39792323112487793, -0.14772173762321472, 0.2972710132598877, -0.7242751717567444, 0.3284463882446289, 0.5573767423629761, 2.498046398162842, 0.4048769474029541, -0.6212624311447144, 1.3280513286590576, 0.40516191720962524, -0.35747095942497253, 0.9199165105819702, -0.48740196228027344, 1.519829273223877, 0.8489220142364502, -0.23085792362689972, 0.059109702706336975, 2.1642282009124756, -0.183842271566391, -0.2990058660507202, 0.800944983959198, 0.2789308726787567, -0.07866953313350677, 0.7682797312736511, -1.9072436094284058, 0.19040286540985107, 1.5187855958938599, 1.444676160812378, -0.6666692495346069, 0.0976346954703331, 0.8901374936103821, -0.17571868002414703, 0.601898193359375, 0.5684327483177185, -1.6900744438171387, 0.8207048773765564, 0.12045550346374512, -0.05868373066186905, -0.22641028463840485, -2.114586591720581, -0.04448578879237175, 1.0339571237564087, 1.3984943628311157, 0.2350405603647232, -0.10707439482212067, -0.3265059292316437, 1.3615119457244873, 2.1674764156341553, 0.10878574848175049, -2.5172324180603027, 0.3216055631637573, 1.6222150325775146, -1.2764140367507935, -0.7487925291061401, -1.4930503368377686, 0.2715160548686981, 2.381467819213867, 1.090132713317871, -1.5634765625, -1.279088020324707, 0.5858190059661865, -0.2501157820224762, -1.6739835739135742, 0.3591102659702301, -0.8359280228614807, 0.4886559247970581, 0.5909976363182068, 0.33104369044303894, -1.0452159643173218, -0.8304569125175476, -0.16881175339221954, -0.5934788584709167, 1.0522581338882446, 1.1718295812606812, 0.2023177593946457, 0.9073431491851807, 1.5246953964233398, -0.9389344453811646, -0.3865225911140442, 1.9431071281433105, -0.16058410704135895, -0.5235275030136108, 0.06046416237950325, 0.49862632155418396, -1.4721148014068604, 0.03436671569943428, 0.9909331202507019, -0.4086025357246399, -0.47618621587753296, 0.9608758091926575, -0.7984333634376526, -1.2547075748443604, -0.17979766428470612, 0.6872251629829407, -0.3144865334033966, -2.6788430213928223, -0.7755390405654907, 0.36003515124320984, 1.6751784086227417, -0.12297989428043365, 2.477046489715576, -1.0433647632598877, -0.5776107311248779, 0.07212820649147034, 0.7706814408302307, -0.32573050260543823, 0.6396445631980896, -0.3071846663951874, 0.47520267963409424, 0.5519726276397705, 0.15930256247520447, 0.20601820945739746, -0.25180619955062866, -0.6979271769523621, 2.9675660133361816, -0.8800317645072937, 1.4161474704742432, 1.084635615348816, 0.06258807331323624, -0.005322332959622145, -0.6994736194610596, 0.25158339738845825, -1.0825525522232056, 0.048628710210323334, 2.0636279582977295, 1.2447062730789185, -1.3224797248840332, 0.21143372356891632, -0.338952898979187, -1.317351222038269, 1.1914833784103394, -1.7460272312164307, 0.5432092547416687, 1.9281970262527466, 1.6585434675216675, -1.1650097370147705, -0.1696958690881729, -0.10597384721040726, 0.009176526218652725, 1.004343032836914, 0.7364439368247986, 1.883129358291626, -0.5166074633598328, 0.525843620300293, 1.3586188554763794, 0.6222531795501709, -1.7283685207366943, 0.24647754430770874, 0.33459338545799255, 0.17694547772407532, -0.813716471195221, 0.8606323599815369, 0.19609688222408295, -0.6951507329940796, 0.6068761348724365, 0.005317931063473225, -0.42928963899612427, -0.5742053389549255, -0.97018963098526, 0.11804354935884476, -0.4934537708759308, 1.1308116912841797, -0.02724168635904789, 0.8051120042800903, -0.747714638710022, -0.4309273362159729, 0.0915476381778717, 0.7692331075668335, 0.9444231390953064, 0.16875523328781128, 0.2735322415828705, 0.6145146489143372, 0.8206976056098938, 1.0712908506393433, -1.6286091804504395, 0.9048022627830505, -0.03265930339694023, 0.1358170509338379, -0.7183337211608887, 0.8129758238792419, 1.2557382583618164, 1.2324249744415283, -0.5643959045410156, -1.490092158317566, 1.3490031957626343, -0.37253084778785706, -1.6615538597106934, 1.0007284879684448, -2.0749824047088623, 1.050241470336914, -0.46829962730407715, 1.9131025075912476, 0.09083381295204163, -0.5659886002540588, 0.14229871332645416, -0.8069026470184326, -0.5561097264289856, 0.4525800943374634, 0.07061924040317535, -1.184828758239746, -0.38904044032096863, 1.831741213798523, -1.301836609840393, 0.5979900360107422, 2.5432162284851074, -0.7658623456954956, 0.7995527982711792, 0.08297143131494522, -0.924187958240509, 0.8326864242553711, 0.7220970988273621, -0.006532507017254829, -0.7371259927749634, 1.0021880865097046, 0.6168938875198364, -0.26255562901496887, -0.8588775992393494, -0.2712271213531494, -0.47417470812797546, -0.5665594339370728, 1.2627906799316406, -0.09822102636098862, 1.4506536722183228, 1.1974811553955078, 1.0030888319015503, 0.5387202501296997, -0.05260651931166649, -0.04327453300356865, 2.0274457931518555, 1.1472090482711792, -0.3281440734863281, 0.7404688000679016, -0.5024834275245667, 0.006288661155849695, -0.9539056420326233, -0.9153535962104797, -1.0307247638702393, 0.37582868337631226, 1.1723538637161255, 0.02076609618961811, 0.3980984687805176, -0.24842536449432373, -0.5006099343299866, 0.3080090284347534, -1.156599760055542, 1.0786287784576416, 0.797358512878418, 0.3862922787666321, 0.026214275509119034, -0.5187779664993286, 1.624704122543335, -0.19201888144016266, -0.5654698610305786, -0.9846094250679016, -0.7030495405197144, -0.1829180121421814, -0.34061750769615173, 0.6730765700340271, -0.43075212836265564, -0.8024366497993469, 0.544248640537262, -0.7410117983818054, 1.816135287284851, -1.2790627479553223, -0.1297074258327484, -0.7912268042564392, 0.3372630774974823, 1.3587348461151123, -0.4485931098461151, -1.5733566284179688, 0.31231144070625305, 0.08465145528316498, 0.6620534658432007, -0.806540310382843, -0.6335685849189758, -0.7507585287094116, -0.5436558127403259, -0.08105389028787613, 1.3859360218048096, 0.4595661759376526, -0.36174413561820984, -1.4391441345214844, 1.4555625915527344, 0.6743341088294983, 1.4155501127243042, -2.76557993888855, 2.406003952026367, -0.9668239951133728, -1.0953117609024048, -0.7090142965316772, -0.5345151424407959, 1.38316810131073, -2.3014845848083496, 0.60566645860672, 1.3253849744796753, 1.1543794870376587, 0.6937899589538574, 0.9759087562561035, -0.3696712553501129, -0.7063874006271362, -0.6422846913337708, -1.7723320722579956, 0.9852070212364197, -0.04352862015366554, 0.3690893054008484, 1.5166877508163452, 1.4003154039382935, 0.5798710584640503, -0.8045787215232849, -1.7750385999679565, 0.8914026021957397, -1.6803473234176636, -0.2875310480594635, -0.12944947183132172, -0.7539559602737427, 0.11502167582511902, 0.9209204316139221, 1.462192177772522, -0.44932612776756287, -1.3749850988388062, -0.22583240270614624, -1.0868841409683228, -1.012609839439392, 0.2343895435333252, 1.0849181413650513, 0.14142581820487976, 0.25158390402793884, 1.137803077697754, 0.6237659454345703, -1.1762542724609375, -0.6020542979240417, -0.4297182857990265, 1.4333062171936035, -1.356695532798767, -0.2201797366142273, -2.8132996559143066, -0.5694693922996521, -0.9687359929084778, -1.0570579767227173, 0.5842776894569397, -0.9264652729034424, 1.3030503988265991, 0.10166919231414795, -0.33322688937187195, -1.136864185333252, 1.6642982959747314, 0.06775465607643127, 1.7431777715682983, 2.4472405910491943, -0.0071297804825007915, 0.7400540113449097, 1.3805993795394897, 0.03191603347659111, -1.6291273832321167, 1.0324780941009521, -0.5313614010810852, -1.1609939336776733, -0.6078178286552429, 1.4257051944732666, -0.9977236390113831, -0.570035994052887, -1.0842159986495972, 1.236306071281433, 0.7357566952705383, 1.679677963256836, 1.02485191822052, 0.5820419788360596, 1.5608524084091187, 1.1494715213775635, 0.3672601580619812, 0.8349547982215881, 2.095437526702881, 0.7937461137771606, -1.6052318811416626, -2.0729002952575684, -0.8813530802726746, 1.1696738004684448, 0.5851617455482483, 0.8589641451835632, -0.02111407369375229, -0.09902986884117126, -0.46104222536087036, -0.12341226637363434, -0.07457125186920166, -0.16472427546977997, -1.165820837020874, -1.4455435276031494, 1.2783756256103516, 0.11787553876638412, 0.38269278407096863, -0.6524879336357117, -0.03742975741624832, 0.42617204785346985, -0.4319530129432678, -0.97300785779953, 0.4710332155227661, 0.5176971554756165, -1.3778643608093262, -1.0892540216445923, 1.1320103406906128, -1.0374737977981567, -1.0606310367584229, -1.0327003002166748, 0.06078599765896797, -0.2834794223308563, 0.6635103225708008, -1.2750194072723389, -0.7974661588668823, -0.12704196572303772, -0.1945815235376358, -1.14289128780365, -0.515985369682312, -0.958632230758667, -0.3734370172023773, -1.337563395500183, 0.9325788021087646, -0.37493470311164856, -0.5037890672683716, -2.8114564418792725, -0.8684823513031006, 0.23336386680603027, 0.7744171619415283, -1.3207452297210693, 1.7027392387390137, 2.1528923511505127, -0.8747212290763855, 1.028992772102356, -1.0543649196624756, 0.30564799904823303, 0.3081786036491394, 0.9366925954818726, 0.2923131287097931, -0.3059340715408325, 0.09755175560712814, 0.6604947447776794, -0.4920262098312378, 0.6940356492996216, -0.6780015230178833, 0.24865855276584625, 0.9805629849433899, -0.599010705947876, 0.3318909704685211, -2.6006758213043213, -0.5510551929473877, 2.1091127395629883, -1.0211163759231567, -1.1007779836654663, 0.8256857991218567, -0.44566619396209717, -0.7165394425392151, -0.5580030083656311, 0.9380759000778198, -0.41745898127555847, -1.8738220930099487, 1.190698504447937, 1.2136366367340088, 0.6127249598503113, -0.19988799095153809, -1.2270900011062622, 1.84199059009552, -2.186030149459839, 1.1879411935806274, 1.0110450983047485, 0.6889472603797913, 0.5024582743644714, 0.967692494392395, 0.060258232057094574, 0.9552602767944336, 0.7526636123657227, 0.11329799145460129, 0.14301598072052002, 0.08062425255775452, 1.4896584749221802, -1.0286037921905518, -0.2420467585325241, 1.39024817943573, -0.7929385900497437, 0.33115077018737793, -0.026223575696349144, -1.5746082067489624, 0.15314048528671265, 0.8761118054389954, -0.19885961711406708, 0.9706213474273682, 0.5160332322120667, 0.34038814902305603, -1.6420249938964844, 0.5721516609191895, 0.3268831670284271, 0.723476231098175, 0.637150764465332, -0.9152162075042725, -0.5645313262939453, -1.2481272220611572, 0.260157972574234, -0.7928549647331238, 1.3375167846679688, 0.9375148415565491, -1.2213608026504517, 0.5833132863044739, -0.5938892364501953, -2.042079210281372, 0.504184901714325, -1.3502929210662842, -0.4763583838939667, 0.6148995757102966, 0.5689529180526733, 2.6950786113739014, 0.5329428315162659, -0.15651758015155792, -0.869663655757904, -2.1979026794433594, -0.565466582775116, -0.14996235072612762, 1.8366305828094482, -0.3998720943927765, -1.6735179424285889, 0.19253495335578918, 0.0036213870625942945, 0.6053935289382935, 0.039994243532419205, -0.8968195915222168, -1.633188247680664, -0.4565337002277374, -0.3079160749912262, 0.10420063138008118, 1.269905924797058, -0.3721967041492462, -0.7529579997062683, -2.842301845550537, 0.6284220218658447, -1.4314746856689453, -0.5252018570899963, -1.0702751874923706, -0.01829805038869381, -0.46201688051223755, -0.45552313327789307, -0.9501836895942688, 0.6270669102668762, -1.3591132164001465, -0.6038661599159241, 1.1146847009658813, -1.2549779415130615, 2.0703518390655518, -0.48235926032066345, 1.3068182468414307, 1.8030160665512085, -0.1764204204082489, 0.3168891966342926, 1.4055402278900146, 0.5634268522262573, 2.6098785400390625, 0.90704745054245, -0.2045241892337799, 1.1814239025115967, 0.15126687288284302, 1.943738579750061, -1.4645087718963623, 1.74696683883667, 2.153154134750366, 1.6933670043945312, 0.09220840781927109, -2.1116673946380615, -0.8583000898361206, 0.0025183765683323145, 1.4986557960510254, 0.2122087925672531, -0.6039311289787292, 0.35084405541419983, 1.4493850469589233, -0.396659791469574, 0.11161253601312637, 0.3007546663284302, 0.530825674533844, 0.9333152174949646, 1.2582539319992065, -1.5784937143325806, 0.5682192444801331, -0.08180428296327591, -0.8881811499595642, 0.2579871118068695, 0.05995199829339981, -0.2320847362279892, 0.5634647607803345, -0.2847959101200104, 0.3079143166542053, 0.11567267030477524, 0.058424126356840134, -0.04124648496508598, -0.3544068932533264, 1.2889114618301392, 0.6385923027992249, 0.3506603240966797, 0.8476215600967407, -0.6396377682685852, 1.4737863540649414, -1.6607996225357056, -0.34706175327301025, -1.7162482738494873, -0.5880106687545776, 0.3386816084384918, 0.026726029813289642, 2.0021116733551025, 0.21899935603141785, 0.43012693524360657, -0.09364918619394302, -1.0531210899353027, 0.7700690627098083, 1.4054220914840698, -1.215336561203003, -1.9747390747070312, -1.7980774641036987, -0.3614145815372467, 0.24501313269138336, 0.6489483714103699, -2.3294382095336914, -1.6275948286056519, 0.10219204425811768, 0.3987622559070587, -0.44573405385017395, -0.6363939642906189, 0.44176405668258667, -0.7302294969558716, -1.7515504360198975, 1.3552918434143066, 0.8942005634307861, -0.3854503035545349, -0.13661161065101624, -0.8614311814308167, 1.6202338933944702, 0.47613435983657837, 0.995370090007782, -0.23942574858665466, -1.065504550933838, 2.2592244148254395, 0.41826096177101135, 1.0548182725906372, 1.4015488624572754, 1.945316195487976, 0.5539669394493103, -1.5028960704803467, 0.5879567861557007, -0.5152419805526733, -0.2181449681520462, 1.0995336771011353, 0.3111712336540222, -0.26042962074279785, -0.0898008793592453, 0.5170881748199463, 1.4328587055206299, 1.3417505025863647, -1.3818999528884888, 0.5695342421531677, 0.36845630407333374, 0.7687098979949951, 0.7721402645111084, 0.6768171191215515, 1.0616059303283691, 0.27103325724601746, 1.2534830570220947, 2.0140042304992676, -0.3167917728424072, -1.1188533306121826, -0.2635906934738159, 0.03972465917468071, -0.226017028093338, -0.8300776481628418, -0.5762572288513184, 0.7747332453727722, 0.9884316921234131, -0.8265299201011658, -0.139034241437912, -1.3828256130218506, -0.6800962686538696, 0.1850569248199463, -0.3867741823196411, -0.0786380022764206, -1.3596209287643433, -0.19490714371204376, -1.6504353284835815, 0.00013156913337297738, -0.048929110169410706, -0.8853621482849121, -0.20696142315864563, 0.24117791652679443, 0.4510173499584198, 2.100470542907715, 1.6325526237487793, -0.3887959122657776, 1.8823401927947998, 0.22525741159915924, 2.567383289337158, 0.44538983702659607, -0.458484411239624, 0.6145851016044617, 0.11002214252948761, 1.4997705221176147, -0.2299540936946869, -0.3425552546977997, -2.28281307220459, -0.23015834391117096, -0.6119527220726013, -0.6017045974731445, 1.150114893913269, 0.8574672341346741, 0.08856529742479324, -0.11615487188100815, -0.7795909643173218, -1.0011056661605835, 0.5890523195266724, 0.30240345001220703, -1.0494349002838135, -0.04469051584601402, -0.29790061712265015, -0.02930304780602455, 1.6472026109695435, -0.7212688326835632, 1.376432180404663, 1.0418819189071655, 0.07901712507009506, 0.942776620388031, 0.4722660779953003, -0.1973218470811844, -1.7729805707931519, -1.30427885055542, -0.9815289378166199, -0.36352044343948364, -0.054072681814432144, 1.3901106119155884, 0.10294313728809357, 1.8254474401474, 1.2857513427734375, 1.182673692703247, 0.021419500932097435, 0.19807332754135132, 0.8140007257461548, -0.18161912262439728, 0.21933718025684357, -1.5421888828277588, 0.7403578758239746, 0.5308229327201843, -0.2990969121456146, 1.5688999891281128, 1.7824854850769043, 0.6248628497123718, -0.5118285417556763, 0.6407539248466492, 0.44766199588775635, 1.0649865865707397, -0.2499508112668991, 0.7080919742584229, 0.8625362515449524, 0.6688116788864136, 0.5646783709526062, 0.15071001648902893, -0.34310775995254517, -0.9720873832702637, -2.038029193878174, -1.9753645658493042, -0.6790555715560913, -0.438469260931015, 1.331963300704956, 0.6341583132743835, -0.7096686959266663, -0.39020413160324097, -0.6512468457221985, -0.7444596290588379, 2.0060503482818604, -0.716858446598053, 0.43394535779953003, -0.24662485718727112, -1.1333153247833252, -0.1635562777519226, -1.5896650552749634, -0.18354760110378265, 0.8544268608093262, -1.0173726081848145, 1.0595321655273438, -0.7143797278404236, -0.5260708332061768, -0.10626175254583359, 1.157955527305603, -0.9016567468643188, 0.43059760332107544, -1.0134848356246948, 0.7707201838493347, 2.1852667331695557, -1.8171277046203613, -0.2999977767467499, -0.683207094669342, -0.5742455124855042, -1.7589274644851685, -0.012406284920871258, -0.8919202089309692, -0.647742748260498, 0.33785480260849, -0.39228326082229614, -0.7034513354301453, 1.8081532716751099, 0.13291718065738678, -0.3097798824310303, -0.05328712239861488, -0.014736060984432697, -0.19554540514945984, 0.5095195174217224, -0.9078402519226074, 1.3310688734054565, 0.08309222012758255, 1.1267696619033813, -0.3164982795715332, 1.7558238506317139, 0.5610620379447937, 0.1521633267402649, 1.6881859302520752, 0.16884037852287292, 0.8876315951347351, -0.22852185368537903, 2.919468879699707, 0.08377578854560852, -0.3836473524570465, 0.3661380112171173, 1.140467882156372, -0.7440341114997864, 1.0835790634155273, -0.3296777606010437, -0.5821068286895752, -0.8769477605819702, 1.4211455583572388, -1.2689577341079712, -0.748453676700592, -0.7017589211463928, -0.15577620267868042, 0.22731958329677582, -0.7027873396873474, -0.7292969226837158, -2.3786776065826416, 1.1443657875061035, 0.3614543378353119, 0.9835585355758667, 0.4515695571899414, -1.6260572671890259, 0.8723514676094055, 1.506181001663208, -1.5523185729980469, 1.0619357824325562, -0.7538400888442993, -0.5417693257331848, -1.2907052040100098, -0.9390026926994324, 0.03111083246767521, 0.5219118595123291, 0.5002248883247375, 0.591801106929779, -0.38051384687423706, 0.5029507279396057, 0.6622864007949829, 1.0036029815673828, 0.5034188628196716, 1.5522226095199585, 0.018276527523994446, -2.2512972354888916, 0.6964269280433655, 0.4672156274318695, 0.9203619360923767, -1.0967479944229126, 0.945237398147583, 0.3901911973953247, -1.0145500898361206, -1.215917944908142, 0.9380866289138794, 2.1807682514190674, 0.7179355621337891, -0.8015305399894714, 0.38648995757102966, -0.9253988862037659, -0.3023184537887573, 0.31113195419311523, -0.2627997398376465, 1.1576828956604004, -1.6311012506484985, 0.11766531318426132, 1.053796648979187, 0.12971587479114532, 0.9989298582077026, -0.3625359833240509, 0.7304772138595581, -1.1201099157333374, 0.02502371184527874, -0.1944478303194046, 0.641162633895874, -0.5927343368530273, 0.06646490097045898, -3.284256935119629, -0.4309968650341034, 0.6251766085624695, 1.2150524854660034, 0.4237642288208008, 0.01595696434378624, 1.8972570896148682, -0.01107372622936964, -0.24103944003582, 0.25171688199043274, -0.11210094392299652, -1.0806587934494019, -1.72001051902771, -1.0994330644607544, 0.2520640790462494, 0.5507439970970154, 0.43627336621284485, -0.6065855026245117, -0.5342257618904114, 0.8109459280967712, -0.7613446116447449, -0.522642195224762, 1.3984103202819824, 0.06351502239704132, -0.6050806641578674, -0.7207634449005127, 1.1504749059677124, -0.06041852384805679, 0.1198909804224968, 0.12987689673900604], [-2.027212381362915, -0.9341920018196106, 0.1623312383890152, -0.14009331166744232, 0.3285822868347168, -0.9462340474128723, -0.9122679233551025, 0.3819066286087036, 1.6351375579833984, -0.99666428565979, -0.07766729593276978, -0.9455220103263855, -1.6270641088485718, -0.11732015013694763, -0.2944765090942383, -2.535865545272827, -0.6901870965957642, -0.778950572013855, -1.6279617547988892, -0.9677894115447998, 1.6044602394104004, -2.403951406478882, -1.5610302686691284, 0.8784468770027161, 0.08144177496433258, 0.35860100388526917, -0.07846565544605255, -0.07175472378730774, -0.7074311375617981, 1.06151282787323, -2.0808329582214355, 0.2071744054555893, -2.2455382347106934, -2.4460597038269043, -0.37843042612075806, 1.4683973789215088, 0.39118993282318115, -0.17123909294605255, 1.605791687965393, -0.2183559387922287, -0.8809282183647156, 0.5051942467689514, -1.5517746210098267, 2.070347547531128, 0.1965944916009903, -0.26413363218307495, -0.7636081576347351, 0.604949951171875, 1.6008516550064087, -0.967093288898468, -0.14592179656028748, -0.07920074462890625, -1.2379614114761353, 0.9143681526184082, 0.02076014131307602, -0.7924002408981323, 0.5331915020942688, -1.1792041063308716, 0.05514712631702423, -0.09214505553245544, -1.2084978818893433, -0.8798929452896118, -0.037182651460170746, -2.1466076374053955, -0.24777935445308685, 0.14123030006885529, -0.10657039284706116, -0.22090710699558258, -1.573668360710144, -0.4306127429008484, -0.24031560122966766, 0.5016021728515625, 0.03472904488444328, 0.9129379391670227, -0.08371604233980179, 0.20163574814796448, 0.6010630130767822, -0.09702654927968979, 1.2662163972854614, -1.828770637512207, -1.460197925567627, 0.9897676110267639, -1.7255154848098755, -0.80106520652771, -0.7359322905540466, 0.23423583805561066, -0.860514760017395, -0.8500996828079224, 2.1634387969970703, -0.2894033193588257, 0.11243361234664917, 0.795775830745697, 0.29338356852531433, -0.42911142110824585, -1.0102176666259766, -0.5094078183174133, 0.44911861419677734, 0.38142523169517517, 0.8502561450004578, 0.8564907312393188, 2.3040876388549805, 1.5057724714279175, 0.228043794631958, 0.34635376930236816, 0.27503302693367004, 0.04640912637114525, 0.21273478865623474, 0.29233118891716003, -0.5137898921966553, -0.9438855648040771, -0.6088061928749084, 0.6476241946220398, 1.2017607688903809, 0.33724331855773926, -0.02124653197824955, 0.7488424777984619, -0.020919345319271088, 0.30468907952308655, 1.087605595588684, -1.6712895631790161, -0.9417702555656433, 2.2981135845184326, -0.020296774804592133, -0.6173885464668274, -0.9967830181121826, -0.3121188282966614, -1.0247116088867188, -0.4678785502910614, -0.3317001163959503, 1.2411999702453613, -0.49225914478302, 0.6925405263900757, -0.8339187502861023, 2.915442943572998, -1.2226670980453491, -0.18801303207874298, 0.10775568336248398, -0.1461515873670578, 0.8209985494613647, 0.7094600200653076, 0.1979427933692932, 1.8468977212905884, -0.5005881786346436, -0.07823818922042847, -0.1785157173871994, 0.10399457067251205, 0.3948119580745697, -0.0030482751317322254, -0.42033132910728455, -0.5288782119750977, -0.2699899971485138, -0.8555606603622437, -0.6250882148742676, -1.1742453575134277, -2.22939133644104, 0.5972004532814026, -0.9340829849243164, 0.9191637635231018, -0.9619873762130737, 0.3725086450576782, -1.2163530588150024, 1.2113075256347656, -0.8376140594482422, -1.5359477996826172, -0.46391114592552185, -1.0458722114562988, 2.7575252056121826, -1.1902928352355957, 0.7474851012229919, -2.43857479095459, 0.7003595232963562, -1.3597906827926636, -0.4361248314380646, -1.084761381149292, 0.5014058351516724, -0.8368133306503296, 0.10647537559270859, 0.43958935141563416, 0.600540041923523, -0.236541748046875, 1.3405547142028809, 1.34033203125, -0.6537145972251892, 0.3868895471096039, 0.9813178181648254, 2.177554130554199, -0.6813920736312866, 0.8310397863388062, 0.23477785289287567, -0.8333495855331421, 0.1207643374800682, -1.4371339082717896, -0.2897849678993225, 2.1738085746765137, 0.8398671746253967, 1.3022541999816895, 0.5116376876831055, -0.25698086619377136, -1.0450465679168701, -1.9264792203903198, -1.2997018098831177, 2.100782632827759, -0.613025963306427, 0.15886269509792328, 1.100669026374817, -1.2359015941619873, 0.4230717718601227, -1.1782562732696533, -0.1438320279121399, -1.3425568342208862, 1.8099004030227661, 0.3219945728778839, 1.4316266775131226, -0.9863604307174683, 0.3442986309528351, -0.925966739654541, 2.736741065979004, -0.3898710012435913, 0.29432278871536255, 0.8611726760864258, -0.4876682460308075, 0.7410628795623779, -0.19759893417358398, 0.12761828303337097, -0.7798181176185608, 0.8284925818443298, 1.3915634155273438, -1.5021522045135498, 0.11049151420593262, -0.36154356598854065, 0.24277399480342865, 0.8380506634712219, -0.18971814215183258, -0.06505190581083298, 0.05205398052930832, -0.11006834357976913, 0.6129403114318848, -1.1340677738189697, -0.10692509263753891, 0.3330438733100891, 1.177038311958313, 1.8176343441009521, 0.23984816670417786, 0.05196448042988777, -1.1739745140075684, -0.9599911570549011, 0.32646462321281433, -0.17915228009223938, 0.5646659135818481, -2.504640579223633, -0.6138640642166138, 0.1332397162914276, -0.27826759219169617, -0.4101885259151459, -0.9725404381752014, -0.42415887117385864, -0.6081746220588684, -0.6420896053314209, 0.02127656899392605, 0.6586654186248779, -0.11691278964281082, -1.1199572086334229, 1.3813036680221558, -0.8141751289367676, 0.025564629584550858, 0.31376218795776367, -0.07471924275159836, -1.3880900144577026, -0.6379398107528687, -1.5663983821868896, 1.4512016773223877, -0.9688313603401184, 1.662215232849121, 0.684537947177887, 0.7354400157928467, -0.00735107995569706, -1.0684832334518433, -0.4094187319278717, -0.19780957698822021, 2.1808722019195557, 0.3692416846752167, -1.117627501487732, -0.23575742542743683, 1.6001993417739868, 0.3845401406288147, 0.4143129289150238, -0.3269796371459961, -1.2527693510055542, 1.7099536657333374, 1.3070573806762695, -1.4987131357192993, -0.029166320338845253, -0.29673635959625244, 1.4028003215789795, -0.05164957046508789, -1.2378193140029907, 1.8292019367218018, 0.17392633855342865, 0.4516894817352295, 0.39787358045578003, 0.4469831585884094, -1.2673333883285522, -2.110452175140381, 0.30491968989372253, 0.7692819833755493, 0.22317110002040863, 1.7046815156936646, -0.30289483070373535, -1.5313303470611572, 1.3355976343154907, 0.9211955070495605, 0.1202150285243988, -2.3328757286071777, 1.6974129676818848, -0.5856672525405884, 0.877159595489502, -0.5979753136634827, 1.8966467380523682, 0.4534366726875305, -0.8925362229347229, -1.9910274744033813, 0.9986798763275146, -1.4978036880493164, 0.13502028584480286, 0.9718312621116638, -1.2258896827697754, 0.9483453035354614, 0.34744203090667725, -0.5195221304893494, -0.046996377408504486, 0.6016677618026733, -0.37293556332588196, 0.8609617948532104, -0.2352665364742279, 0.10191310942173004, 0.6454718708992004, 1.0504816770553589, 0.2509397864341736, 0.6023409366607666, 2.4775702953338623, 0.5493401885032654, -1.7853031158447266, 1.5268323421478271, 0.10316219925880432, 1.0290577411651611, 0.07990934699773788, 0.327222615480423, 1.2635761499404907, 0.482856422662735, -0.7311751842498779, 1.3153042793273926, -0.8254538178443909, 0.9323755502700806, 0.7223562598228455, -0.06054180487990379, 0.03390863537788391, -1.389668345451355, -0.5937627553939819, 3.205963373184204, -2.079542398452759, 1.0635472536087036, -0.9651137590408325, -0.4164019525051117, -2.0651156902313232, -1.2045954465866089, -0.22280645370483398, 0.5560035705566406, 1.1842478513717651, 1.4312896728515625, -0.03794629126787186, -0.8695124387741089, 1.1311495304107666, 0.20938099920749664, -1.2515171766281128, 1.9822728633880615, 1.0267245769500732, 1.3512787818908691, 0.06330197304487228, -0.6693089604377747, 0.772761344909668, -0.36459454894065857, 1.6310697793960571, -0.3562620282173157, -0.3460312485694885, 1.3453270196914673, 1.392034649848938, 0.4175082743167877, 1.1709165573120117, 0.4505607485771179, -0.6605130434036255, 1.2209155559539795, -1.1339348554611206, 0.0009437286062166095, -0.29834166169166565, -0.5797454714775085, -0.8067882061004639, -0.0025437704753130674, -0.4939843714237213, -0.19959545135498047, -0.4785611927509308, -0.8081220984458923, -0.4288569688796997, 0.5111197233200073, -0.1210629940032959, -1.089490532875061, -0.4273114502429962, -1.2443920373916626, -0.38231974840164185, 0.1707940548658371, -0.3958114981651306, -0.11590977758169174, 1.1099838018417358, -0.4472960829734802, 0.3737143278121948, -0.8071562051773071, 1.4859631061553955, -0.1706244796514511, 0.013789039105176926, -0.8438760042190552, 0.7096531391143799, 0.8703790307044983, 1.3859257698059082, -0.49177494645118713, 0.5606589317321777, -1.8450952768325806, -1.401288390159607, 1.214013934135437, 1.2456632852554321, 0.03764084726572037, 0.1329507827758789, 2.3995046615600586, 0.7421968579292297, 1.1502612829208374, 0.6301984190940857, 0.014956195838749409, -1.1509525775909424, 0.48519834876060486, 0.09253428131341934, 0.11361800879240036, 0.5380305647850037, 0.024907555431127548, 1.543053150177002, -1.1598944664001465, 0.5818834900856018, -0.8866407871246338, -0.23223866522312164, -1.200522780418396, -0.02542625181376934, 1.042023777961731, 0.5624534487724304, -0.11412853002548218, 1.7014555931091309, -0.921849250793457, 1.7383953332901, -0.22968348860740662, 1.2951557636260986, 0.10659696906805038, -0.32369264960289, 1.6846115589141846, -0.6665053367614746, 0.4796401262283325, 0.8210223317146301, 1.1591888666152954, -0.9588465690612793, 1.0529173612594604, -1.227097988128662, 0.2746734619140625, 0.6995095014572144, 0.6404300332069397, 0.19682978093624115, 0.09437960386276245, 0.4468640387058258, -0.5522560477256775, 1.2995673418045044, 1.1410577297210693, -0.011157919652760029, 0.8864271640777588, -0.2243817150592804, 0.5473150014877319, -0.44487372040748596, 0.5905905961990356, 0.6956019997596741, -0.14976130425930023, -1.5460731983184814, 1.5892536640167236, 0.9340658187866211, 0.7584746479988098, 1.0006086826324463, 0.23121005296707153, 0.830052375793457, 0.55910724401474, -1.2046726942062378, -0.8785898089408875, -1.5422518253326416, -0.915414571762085, -0.6311972141265869, -0.7890686988830566, -2.3125288486480713, -0.18966059386730194, -1.0667154788970947, -1.452322244644165, 0.22705276310443878, 1.0536631345748901, 1.237212061882019, 0.6569485664367676, 0.8788581490516663, 2.4506337642669678, -0.9510060548782349, 1.009832739830017, -0.5386055707931519, 0.5977813005447388, -0.04224016144871712, 0.25056689977645874, -0.12294016033411026, -0.3882778584957123, -0.7953976988792419, 1.150076150894165, 0.9190258383750916, 0.8010358810424805, -0.038722388446331024, -1.3975317478179932, -0.04325640946626663, 1.115965723991394, -1.424006700515747, 1.1360538005828857, 0.6911733150482178, 0.06100064516067505, -0.889665424823761, 1.288301944732666, -1.4308891296386719, -0.2882827818393707, -0.6696733832359314, 0.09448067843914032, 0.36248916387557983, 0.16903796792030334, -0.8438453078269958, -0.6414136290550232, 0.2036561220884323, 0.4252477288246155, 2.2147223949432373, 1.6401227712631226, 0.03560815379023552, 0.35864129662513733, 1.033694863319397, 0.8425357341766357, 0.04891345649957657, 1.0041590929031372, 0.2957092821598053, -0.9775871634483337, -2.3316140174865723, -1.8250415325164795, -0.9816120266914368, -0.5992862582206726, -0.2887282967567444, 1.030975341796875, -1.283775806427002, -1.6159449815750122, 0.24609246850013733, -0.17895928025245667, -0.6057880520820618, -0.5067340135574341, -2.7971549034118652, -1.1021215915679932, 0.5132483839988708, 0.5062865018844604, 1.6270679235458374, 1.0900311470031738, -0.2150314748287201, -1.3150928020477295, -0.35220199823379517, 0.8195850253105164, 1.3168208599090576, 3.026092529296875, 2.4086363315582275, 1.177417278289795, 0.01156081072986126, -0.4421345889568329, -0.32970213890075684, -1.981069803237915, 0.3753800392150879, -0.30723485350608826, -0.7831155061721802, -0.04717518389225006, -0.20168358087539673, -1.012610912322998, 0.6199302673339844, 0.41152116656303406, 0.15012788772583008, 1.5329487323760986, 1.2727231979370117, 0.13933165371418, -0.923058032989502, -1.286644458770752, 0.48109856247901917, -0.013460476882755756, 1.3760037422180176, -0.36217865347862244, 0.6772653460502625, -0.37448427081108093, -0.15357880294322968, 1.564708948135376, -0.18636827170848846, -0.33861157298088074, 0.3587627708911896, 2.6022090911865234, -1.0301438570022583, 0.6757077574729919, -1.6188623905181885, -0.24164535105228424, 0.41199928522109985, 0.5468494892120361, 1.4149963855743408, -1.230760097503662, -1.0033336877822876, -0.09598445892333984, 0.14112192392349243, -0.17902575433254242, -0.320778489112854, -0.6755789518356323, 0.3560774326324463, -0.37905871868133545, -0.49127197265625, 1.2434985637664795, 1.1302579641342163, 0.5928386449813843, 0.6062266230583191, 0.8706445693969727, 2.159485101699829, 0.33880969882011414, -0.16480796039104462, -0.023712335154414177, 0.3771873116493225, -0.8234540820121765, -0.045484863221645355, -1.0268181562423706, -0.3195887506008148, 0.29128459095954895, 0.052754711359739304, -1.3713992834091187, -1.216372013092041, -0.6352856159210205, 0.11636954545974731, -1.3655940294265747, -2.079634428024292, -0.2536643445491791, 0.642122745513916, 1.202610731124878, 0.6180893778800964, 0.422875314950943, 0.2403881996870041, 1.055416464805603, -0.22807210683822632, -0.39587560296058655, -1.114574909210205, -0.8730965256690979, -0.7012988328933716, -1.2678824663162231, 0.1526680737733841, 0.5376710295677185, -0.8723023533821106, 0.5831720232963562, -0.3009359538555145, 1.6660369634628296, -0.5132409334182739, 0.05136486142873764, 0.3599173426628113, 0.6944476962089539, 1.347686529159546, 0.19965550303459167, 2.4719505310058594, -0.37019458413124084, -0.17555885016918182, 2.003553867340088, -0.08876825124025345, -0.19892652332782745, 0.5691561698913574, -0.923289954662323, 1.1330902576446533, -0.841180145740509, -1.466753363609314, -0.9000263810157776, 0.2863824665546417, 0.9973633885383606, 0.7245398759841919, 1.0884308815002441, -0.2008838653564453, -0.5300279259681702, 0.9864057302474976, -1.516984462738037, 0.18107011914253235, -0.2561492323875427, 0.2333032488822937, -2.200355291366577, -1.0095943212509155, -0.08072855323553085, 0.05688370019197464, -2.21254301071167, 0.5215395092964172, 0.43264421820640564, -0.029161810874938965, -0.6775400638580322, -0.5382024049758911, 2.0777831077575684, -0.10058239102363586, 0.11494950950145721, 1.5324167013168335, 0.5151960849761963, 0.06159069389104843, -1.6303777694702148, 0.673778772354126, -1.2563815116882324, -0.7826854586601257, 0.7037959098815918, -0.18038515746593475, -0.7240720987319946, 0.484393835067749, 1.3947457075119019, 0.6846845149993896, 0.24776729941368103, -0.2991292476654053, 0.6377152800559998, 0.8082361221313477, -1.2503913640975952, 1.123095989227295, -2.0112814903259277, 0.8900593519210815, -0.622003436088562, -0.559481143951416, 1.1694685220718384, 0.894469678401947, 0.13508516550064087, -0.4954271912574768, -0.3325272500514984, 1.176892638206482, -0.8723781108856201, -1.4135198593139648, -0.01835356280207634, 0.1886487454175949, -0.20585669577121735, -1.7348884344100952, 0.804578959941864, -1.024071455001831, -0.6617374420166016, 0.9203270077705383, 0.049380503594875336, -1.117592453956604, 0.5319176316261292, -0.05128491297364235, 0.88917076587677, 0.7523844838142395, 2.3152382373809814, 0.7436093091964722, 1.298264741897583, -0.03401566669344902, 0.0068484144285321236, -0.5896354913711548, 0.009946132078766823, 0.05950472503900528, 0.48681387305259705, -2.383511543273926, 0.4375186860561371, -0.7747373580932617, 1.4096678495407104, -1.0288254022598267, -0.293117493391037, -0.8972139358520508, 0.6258694529533386, -0.8504767417907715, -1.1127551794052124, -0.8027114272117615, 0.14440327882766724, -0.07291163504123688, -1.1697486639022827, -2.328019618988037, 0.42892372608184814, 0.5825205445289612, -1.320197582244873, -0.32683050632476807, 0.02896573208272457, -0.9846957921981812, -0.43815484642982483, -2.100231647491455, -1.1913836002349854, -0.1822202354669571, -0.901846170425415, 0.6091117858886719, 0.40482065081596375, -0.12841932475566864, 0.206404447555542, 0.7491430044174194, -0.2818125784397125, 0.8058107495307922, 0.026576289907097816, 0.6500924825668335, -1.2569669485092163, -1.1527516841888428, -0.04822390154004097, -1.1665889024734497, 0.28922784328460693, -0.16264258325099945, -0.14399699866771698, 1.6367785930633545, -2.2355682849884033, 0.909561276435852, 0.16912703216075897, 1.4361310005187988, -0.9873172044754028, -1.3316779136657715, -0.7092940807342529, -0.77528977394104, 0.7953953146934509, -0.5834044218063354, -0.07560476660728455, -0.08286639302968979, 0.011121728457510471, -0.08978110551834106, -0.6497506499290466, -0.22232380509376526, 0.9509828686714172, 0.47756266593933105, -0.43010538816452026, 0.13220906257629395, 0.8219572305679321, -2.051490306854248, -0.7866310477256775, -0.11124783754348755, -0.3400719165802002, 0.9916526079177856, -0.09361160546541214, -0.25119152665138245, -1.7579362392425537, 1.860202431678772, 0.584044873714447, -1.6451988220214844, -0.7140471339225769, -1.1829496622085571, 1.417583703994751, -0.13399282097816467, 0.027506273239850998, -0.9786738753318787, -3.620698928833008, 0.07267102599143982, -0.658670961856842, -0.30119141936302185, -0.766673743724823, -0.5438216328620911, -0.9722294211387634, -0.11354216188192368, -0.017026865854859352, 1.0107372999191284, 0.5721849799156189, -0.5464080572128296, -0.9399558901786804, 2.2400403022766113, -0.24988484382629395, 1.7684968709945679, 2.710561752319336, 0.27362990379333496, -1.3948848247528076, -2.086296796798706, -0.011498148553073406, 0.5920909643173218, -1.6002486944198608, -1.6945258378982544, -0.2340497076511383, -0.8471585512161255, 0.5101379156112671, -0.6350849270820618, 0.4698374569416046, 0.14784175157546997, 0.6117638349533081, 0.3205258846282959, 0.17429493367671967, 0.2599084973335266, -0.39861592650413513, -0.18850429356098175, -1.701245665550232, 1.6313962936401367, 0.40677425265312195, -0.43693363666534424, -2.942837953567505, -0.8702319264411926, -0.03826282173395157, 1.7274394035339355, 0.5647140145301819, 0.19925521314144135, 0.6181826591491699, 1.8630943298339844, -1.8745462894439697, 0.03250398859381676, 0.7251542806625366, 2.1413280963897705, 1.4679688215255737, -1.1031121015548706, 0.8972423672676086, 0.4870873987674713, -0.2147337794303894, -2.061053514480591, 0.5971361398696899, -0.30752718448638916, 0.8250946998596191, -1.3095498085021973, -1.077924132347107, 0.8157143592834473, -1.9340670108795166, 0.052573058754205704, -0.07335861027240753, -0.38840603828430176, 0.6909445524215698, 1.86421537399292, -0.27590781450271606, -0.15513937175273895, -0.08556633442640305, 0.1457226276397705, 0.48958301544189453, 0.9033852815628052, 0.38702788949012756, 0.40980762243270874, -0.27068737149238586, 2.0674612522125244, -0.6163947582244873, -0.8542797565460205, -0.3067880868911743, -0.7149297595024109, -1.4387860298156738, 0.47429782152175903, -0.9466902613639832, 2.139260768890381, -0.9592284560203552, -0.4330644905567169, -1.7009860277175903, 1.5177192687988281, -0.8264942765235901, -1.5076568126678467, 0.043925829231739044, -1.2532278299331665, 0.2092140167951584, -0.13762395083904266, 0.5487076044082642, 1.856957197189331, 0.6605382561683655, -0.9429166913032532, -2.228142023086548, 2.021209955215454, 0.5419073104858398, -0.7942624688148499, 0.059666186571121216, 0.29946082830429077, -1.1669613122940063, 0.5944716930389404, 0.9768342971801758, 0.1508990228176117, 0.3643226623535156, -0.6341388821601868, 0.03621792420744896, -1.9106897115707397, -0.4949197471141815, 3.059422254562378, -0.33777114748954773, -0.3411176800727844, -0.4693877100944519, 0.8080737590789795, 0.41071340441703796, 1.2425713539123535, -0.5533687472343445, -0.495624303817749, 0.1726578027009964, -0.38218605518341064, -0.9477357864379883, -0.8006447553634644, -0.5571761727333069, 2.095780372619629, -0.7087429761886597, 2.0212152004241943, -0.4369194805622101, -1.606027603149414, 0.736763060092926, 0.5220363736152649, 1.809346079826355, -0.9139049053192139, 1.8420209884643555, -2.0461184978485107, -1.877834439277649, 0.8317510485649109, 1.7227834463119507, 0.9444320797920227, -1.1923905611038208, 0.6064218878746033, 1.4614946842193604, 1.5343072414398193, 1.2418264150619507, 1.5146108865737915, 0.42718103528022766, -0.9929362535476685, 1.3749926090240479, 1.224215030670166, 1.1164251565933228, -2.5736958980560303, -0.7832720875740051, 1.581919550895691], [-0.658768892288208, 1.5736106634140015, -1.1569560766220093, 0.7277995347976685, -2.351349353790283, 0.9372887015342712, 0.3313874900341034, -0.759551465511322, -0.2676067650318146, -0.026510875672101974, -0.7946999669075012, -0.7072619795799255, 1.8601957559585571, 0.6076949834823608, -0.17258617281913757, 1.178291916847229, 0.5507257580757141, -1.4939521551132202, -1.3898875713348389, 0.375914067029953, -1.0725544691085815, 0.4629993736743927, -0.42129236459732056, -1.0342371463775635, -0.6115350723266602, -2.2599844932556152, 1.4191579818725586, 0.32004210352897644, -0.022232361137866974, 2.036003589630127, 0.4306911528110504, -1.54582679271698, 0.9758988618850708, 0.9839149117469788, -0.20373816788196564, 1.8569754362106323, -0.44246014952659607, -0.18034951388835907, 0.7082545757293701, -0.08021850883960724, -1.1736434698104858, -0.32642862200737, -0.2403283417224884, 1.718505859375, -0.01979409158229828, -1.7255964279174805, -0.5866593718528748, -0.6165271997451782, 0.07135198265314102, -1.0177040100097656, -1.1110293865203857, 0.6221227049827576, -0.9072040915489197, 1.1848671436309814, 0.11831501126289368, -0.15473966300487518, -0.2751396894454956, 0.3618718087673187, 0.3539949059486389, -1.0088493824005127, 1.1603609323501587, -0.39099037647247314, 0.6308029890060425, -0.04234017804265022, 1.935436725616455, -0.09258327633142471, -0.7618842124938965, 0.028369568288326263, 1.0260379314422607, -0.17037205398082733, -1.7200496196746826, 0.05855533480644226, -0.5587982535362244, 1.399627447128296, 0.39153632521629333, 0.07646948099136353, 1.2559279203414917, -0.0061791446059942245, -1.997377634048462, -0.30293008685112, -0.5850195288658142, 0.27315500378608704, -1.2279170751571655, -0.13719524443149567, 0.045530010014772415, -0.5206542015075684, -0.9829481244087219, 0.9300185441970825, -1.1842392683029175, 1.3531726598739624, 0.6524451971054077, 0.3110285997390747, -0.5101561546325684, -0.10557679086923599, -0.04517906531691551, -0.7514428496360779, -0.7414149045944214, 0.35593363642692566, -0.8640756011009216, 0.19303935766220093, -0.6290757060050964, -1.0289266109466553, 0.6574608087539673, -0.5471183657646179, -0.9737839698791504, -0.5431044697761536, 0.5984399318695068, -0.7877867221832275, -0.27701741456985474, -1.614257574081421, 0.10444695502519608, -0.18101103603839874, -1.1680567264556885, 0.4949958622455597, 0.524316668510437, 0.8256294131278992, -0.6580135226249695, -1.249672770500183, 1.3574312925338745, -1.9291996955871582, 0.07795774191617966, 0.0272387582808733, 0.36712005734443665, 0.04334484413266182, 0.17813526093959808, -0.3127002418041229, 0.26183655858039856, -0.16345752775669098, 0.29724714159965515, 2.6083340644836426, -1.3741447925567627, -2.562758207321167, 0.2674850523471832, 2.31701922416687, -0.9940633773803711, 1.3243528604507446, 0.16651329398155212, 0.5816178917884827, -0.4250171482563019, 1.572767734527588, -0.03125949576497078, 1.6077295541763306, 2.612030506134033, 2.2648892402648926, 1.2011948823928833, -1.817704200744629, 0.36810463666915894, 0.9638303518295288, -0.1667633205652237, -0.3149973154067993, -0.7867739200592041, 2.5183470249176025, 0.5267429351806641, -0.22660627961158752, -0.1754017472267151, 0.7384726405143738, 0.7925978899002075, -0.8380304574966431, 0.09688316285610199, 0.51228266954422, 0.14531254768371582, 2.2104742527008057, 0.5031144618988037, 0.32613107562065125, -2.98000168800354, -0.4031001031398773, 1.3732489347457886, 1.8934383392333984, 0.7223225831985474, 0.9984505772590637, -0.42270201444625854, -0.6774265170097351, -0.4939998388290405, 1.0560764074325562, 0.1760922223329544, -1.0447964668273926, 1.275813341140747, -0.602713942527771, -0.9368394017219543, -0.11091875284910202, 0.6191571950912476, -0.28300079703330994, 0.7869701385498047, 0.008070570416748524, 1.2151970863342285, -0.05127962306141853, 0.8497847318649292, -0.3601687550544739, 0.59434574842453, 0.14307549595832825, -0.9760507941246033, 1.4475579261779785, -0.2564167082309723, 0.43874332308769226, -0.40097692608833313, 0.12702834606170654, 0.041530389338731766, -0.46849915385246277, 0.20895014703273773, 0.07976911962032318, 0.6974712610244751, -0.11180250346660614, 0.9050192832946777, -0.20821282267570496, -0.2506377100944519, -0.14229199290275574, -0.2526819705963135, 0.05511920154094696, 0.5387238264083862, 0.35350728034973145, -0.7529898285865784, -1.451768398284912, -1.5982955694198608, 1.8362045288085938, -1.1678017377853394, -0.1304749995470047, 1.8713114261627197, 0.603640615940094, 0.28897082805633545, -0.49275633692741394, -0.4868835508823395, 0.23759685456752777, 0.7724875807762146, 0.8974559307098389, 0.7035151720046997, -0.9195660352706909, -0.4055195450782776, -0.05745723471045494, 0.3841567635536194, 1.657518982887268, 0.07957105338573456, -0.3548906445503235, 0.42700690031051636, 0.3018346130847931, 0.9580093026161194, 0.033744584769010544, -0.3736095130443573, -1.3347969055175781, -0.22392044961452484, 0.2631986141204834, -0.22885942459106445, 0.7622123956680298, 0.47134414315223694, 0.5261385440826416, -1.5602353811264038, 1.2442463636398315, 0.6878668069839478, -2.4523582458496094, -0.575381875038147, -2.166987895965576, -0.728330671787262, -0.9632562398910522, -1.2461711168289185, 0.6307991147041321, 0.8861387372016907, -0.3409644067287445, -1.1196757555007935, 1.1473451852798462, -0.6524898409843445, -0.13749736547470093, 0.5081327557563782, -0.12070868164300919, -0.5705610513687134, -0.19587171077728271, 0.37669995427131653, 0.5965849757194519, -0.35233980417251587, -1.4504460096359253, 0.40319666266441345, 0.11013928055763245, 1.142698884010315, 0.18150921165943146, -0.4988216459751129, -0.703873872756958, 0.635614812374115, -1.7996245622634888, -0.0013332095695659518, -1.2000751495361328, 0.3647644817829132, 0.06119566783308983, 0.03784039616584778, 0.4007515013217926, 1.0829051733016968, 1.2447999715805054, -0.4594736397266388, 1.154165506362915, 0.14547614753246307, 0.4099932610988617, 0.1599433571100235, 0.2948031425476074, -0.5807047486305237, -1.113982915878296, -0.7057541012763977, 0.42887380719184875, 1.7250757217407227, -0.873630940914154, 1.055406928062439, -1.3306491374969482, -0.1641256958246231, 0.9573988318443298, 1.068608045578003, 0.044956497848033905, 2.184391975402832, 0.24962984025478363, -0.1659654974937439, -1.5248804092407227, 0.05227899178862572, -0.22344990074634552, -0.40476733446121216, 0.712790310382843, -0.454491525888443, -0.8344030380249023, -0.6103084683418274, 0.8218732476234436, -0.1933537870645523, 1.2053086757659912, 0.9111830592155457, -0.2378670573234558, -2.217400550842285, 0.35716503858566284, 0.31110310554504395, 1.1421316862106323, 1.7007577419281006, 0.5924357175827026, -0.5677471160888672, -0.7514703273773193, -0.3577117919921875, -0.2968403398990631, -0.398911714553833, -0.10291995853185654, 2.0327017307281494, -1.5507525205612183, -2.6383752822875977, -0.22916731238365173, -0.40703442692756653, -1.3738559484481812, 1.1497869491577148, 0.506401002407074, 1.1940406560897827, 1.0516992807388306, 0.7415699362754822, -0.19066542387008667, -0.7185505628585815, 0.23186032474040985, 0.8373229503631592, -1.3581680059432983, -0.878621518611908, -0.6476312875747681, -0.5712485313415527, -1.622131109237671, -0.3641307055950165, -0.3797239661216736, 0.18879522383213043, -1.627343773841858, -1.3105597496032715, 0.6053276062011719, 1.2898081541061401, 1.9317002296447754, 1.2131742238998413, -0.3397061228752136, 0.8666900992393494, -1.4044997692108154, 1.0684943199157715, -0.5328255891799927, -2.537933588027954, 1.5434012413024902, -0.09347311407327652, 1.1698561906814575, -0.172825425863266, 1.1730877161026, 0.8602638840675354, 0.6530765295028687, -1.725364089012146, -0.8757296800613403, -1.8406927585601807, 0.5048893094062805, -0.8300695419311523, 0.9495578408241272, -0.604954183101654, -0.18710912764072418, 0.6404600739479065, -1.044277310371399, 0.23965179920196533, -2.728862762451172, -0.8521019220352173, 0.8720730543136597, 2.5571258068084717, -2.0882885456085205, 1.5217965841293335, 1.3571852445602417, -0.45561298727989197, -0.9500408172607422, -0.20630428194999695, -0.2386072725057602, 0.3629288673400879, -1.5359532833099365, 0.08322611451148987, -0.5779122114181519, -0.3545258045196533, 0.6824144124984741, 0.373078316450119, -2.2268710136413574, 0.10328089445829391, 0.018634850159287453, 0.606789767742157, 0.17617100477218628, 0.19522657990455627, -0.3507850766181946, -0.0013409251114353538, 0.825894832611084, -1.4744540452957153, -0.28767332434654236, -0.40422144532203674, -0.7377128601074219, -2.2359137535095215, 1.1253101825714111, -0.409514844417572, 1.4818035364151, 2.4339089393615723, -1.0377752780914307, 0.7950567603111267, 0.015563816763460636, 0.5445554852485657, 0.6663075685501099, 0.7808454632759094, 0.5341863632202148, 0.11867372691631317, -0.9454049468040466, -0.5827914476394653, -1.2167656421661377, -1.370613694190979, 0.005181040149182081, -0.7786538004875183, -1.810171127319336, -1.964006781578064, -0.5086473822593689, -1.069506049156189, 0.4001084268093109, 0.00917451549321413, 0.45958614349365234, 1.4014400243759155, -0.8479149341583252, 0.4230848252773285, -0.641331672668457, -0.7538557648658752, -0.15447862446308136, -1.6839351654052734, 0.2999626696109772, -0.002918354468420148, -0.2946690320968628, 0.15233397483825684, -0.7402974963188171, 0.5092693567276001, -2.2783889770507812, -1.3800086975097656, 0.24888500571250916, -0.4339251220226288, 0.36014649271965027, -2.296839475631714, -0.3937016725540161, -0.5773389339447021, -0.1127827987074852, -1.543651819229126, 1.0674222707748413, 0.38631412386894226, 0.27677470445632935, 0.24464154243469238, -0.4681304693222046, -0.8835166692733765, 1.2114351987838745, 0.8404465317726135, -0.14954544603824615, -1.0798262357711792, -0.28368544578552246, 1.2943426370620728, 0.5570475459098816, 1.6265537738800049, 1.31455659866333, 1.3896420001983643, 0.5288125276565552, -0.06717730313539505, -0.20766198635101318, -0.9772506952285767, -1.041608452796936, 0.36684665083885193, -0.7441472411155701, -2.123300313949585, 0.505014955997467, -1.257225513458252, -0.43036022782325745, -0.5639366507530212, -0.722574770450592, 1.2295690774917603, 0.26994335651397705, -1.8434950113296509, 2.7370662689208984, -0.07033763825893402, 1.1863768100738525, 1.021872639656067, 0.08144978433847427, -1.5179229974746704, -2.158576250076294, 1.9094904661178589, -1.620724081993103, 0.909949779510498, -1.7120085954666138, -1.1844596862792969, 0.7879543900489807, -0.7483347654342651, -0.13426312804222107, -0.7697803378105164, -1.2070896625518799, 1.006969928741455, 0.07320594787597656, -0.3064851760864258, -1.082406759262085, -1.4402134418487549, -0.6861987113952637, 0.1251240372657776, 0.3522656559944153, 0.25763100385665894, 0.6081559062004089, -1.7393231391906738, 0.8501214981079102, -0.7445382475852966, -0.6132389903068542, 1.2152018547058105, 0.5991712808609009, -0.38993555307388306, -0.017828045412898064, -1.1548126935958862, -0.6545117497444153, 0.364888072013855, -0.2853086292743683, 0.9329594969749451, -0.572212278842926, 0.2028149664402008, -1.7486326694488525, -0.17037571966648102, 0.13471238315105438, -0.3493090271949768, -1.9412747621536255, 0.05482880026102066, -1.9245672225952148, -0.5817552208900452, 0.3904614746570587, -0.21724645793437958, -1.088624119758606, 0.21149630844593048, -1.96131432056427, 0.8769636750221252, 0.29668617248535156, -2.9732208251953125, -0.03125297278165817, 0.8500313758850098, 0.18779553472995758, -0.3218362331390381, -1.3828412294387817, -1.6074751615524292, -0.00871368870139122, -0.008957860060036182, 1.2156814336776733, -1.2808668613433838, 1.47394859790802, 0.8803182244300842, 0.48482510447502136, 0.1614200621843338, -0.546960175037384, -0.9608049392700195, 0.29831016063690186, -0.8729645609855652, 0.4387038052082062, 1.5920765399932861, 1.5348713397979736, 1.250909447669983, 0.30078044533729553, -0.20390138030052185, 1.3103903532028198, 1.0961518287658691, -0.8521706461906433, -0.9645744562149048, 0.1042686253786087, 0.22139501571655273, -1.016409158706665, -1.7883620262145996, -0.3125903308391571, -0.4102981686592102, 0.36049821972846985, -1.1955653429031372, 0.33230432868003845, -0.44967833161354065, 0.40113401412963867, 1.5439367294311523, 0.5298107862472534, 0.01527070440351963, 0.9517738223075867, -1.3908512592315674, -0.38995492458343506, -0.03356734290719032, -0.23289071023464203, -0.5965681672096252, 1.8816672563552856, 0.26629960536956787, 0.5805485248565674, 0.17060905694961548, -1.625807762145996, 0.7728652358055115, 0.7248308658599854, -0.11601574718952179, -0.3337568938732147, 0.15682414174079895, -1.7245638370513916, 1.4325133562088013, -0.5080684423446655, 0.26053017377853394, 0.09320124238729477, -1.2781847715377808, 0.21668002009391785, -0.04102155938744545, -0.016621243208646774, 0.4611313045024872, -1.0131562948226929, -1.324414610862732, -0.07369828224182129, 1.0645596981048584, 0.6985719799995422, 0.5986456274986267, -0.9276491403579712, -1.2297550439834595, 0.6791383028030396, 0.8954679369926453, 0.11011315137147903, 0.06070429086685181, -0.28917720913887024, -1.6206974983215332, 0.3858399987220764, 0.6779938340187073, 0.4709714353084564, 0.11184313893318176, 1.0559196472167969, -1.0238896608352661, -1.6472992897033691, 0.8453923463821411, -0.05476631596684456, 0.7935878038406372, 0.08104293793439865, 0.6225787997245789, 1.300862431526184, 0.2442784607410431, -1.3318493366241455, 2.1120336055755615, 0.06078954041004181, 0.9010428190231323, -0.13257642090320587, -0.7590569257736206, -0.3290368616580963, -2.727506399154663, 0.43045875430107117, -0.6675900816917419, 1.6955374479293823, 0.6770962476730347, 1.315592646598816, 0.671169102191925, -0.6002954244613647, 0.1585201770067215, 1.1145414113998413, -0.12652269005775452, 0.2751530706882477, 0.1552574336528778, 0.5146524310112, 0.08884449303150177, 1.401836633682251, 0.923413097858429, 0.19823789596557617, 0.49360939860343933, -0.517559289932251, 0.03729194030165672, 0.4431071877479553, 0.42921924591064453, -1.9593127965927124, -0.5102509260177612, -1.045566439628601, 2.187586545944214, 1.337931752204895, 0.6805775761604309, -0.5279231071472168, -0.042756855487823486, 0.4039519131183624, 1.4485180377960205, -1.105892300605774, -0.8206778168678284, -1.1733354330062866, -0.983112096786499, -0.6082732677459717, -0.29245081543922424, 1.0451300144195557, -0.6338350176811218, 0.19523076713085175, -0.2718544602394104, -1.4980827569961548, 1.0838887691497803, 0.8963630199432373, -0.01821439526975155, -0.33607590198516846, 0.06870440393686295, 0.6374863386154175, -0.6942564249038696, -0.6397459506988525, 0.3507346212863922, -1.0298813581466675, -1.4107093811035156, 0.19509592652320862, -0.5584399104118347, -1.6027098894119263, 1.1781814098358154, 0.2846534848213196, -1.1859697103500366, -1.4769221544265747, -0.8090218305587769, -0.37174859642982483, 0.8590663075447083, -0.9132673144340515, 0.09743029624223709, -1.54108726978302, 0.28396907448768616, 0.4403296411037445, 1.3822633028030396, 0.7727031111717224, -1.1075702905654907, 0.003181760897859931, 0.6272437572479248, -0.6664355993270874, -0.5597661733627319, -0.07747805118560791, -0.011088944040238857, -1.42384672164917, 0.2035369724035263, 0.8205453753471375, -0.3241545557975769, 0.3383035361766815, 0.4410938024520874, 0.05522938072681427, -0.6277346014976501, -0.6039618253707886, 1.5010719299316406, 1.26460862159729, -1.5722887516021729, -0.584929347038269, 0.42614251375198364, 0.13448360562324524, 2.1166398525238037, -0.9965283274650574, 0.4995134472846985, 0.22804121673107147, 0.09823150932788849, -0.8275250196456909, -0.9791752696037292, -0.3371308445930481, -2.012505054473877, 1.3691197633743286, 0.1703682541847229, 1.0966222286224365, 0.24468174576759338, 2.085779905319214, 0.29134005308151245, 0.1553463488817215, 1.7023013830184937, 0.7955161929130554, 2.4658894538879395, 0.1773536205291748, 0.1740524023771286, 0.8103366494178772, 0.25409215688705444, 0.27990785241127014, 0.6620864272117615, -1.1641443967819214, -0.09722158312797546, 0.5763930082321167, -0.789383590221405, -1.1872994899749756, -2.35406756401062, -0.48283788561820984, -0.4180268943309784, -0.5583347678184509, 1.4861868619918823, 1.9287461042404175, -0.905677318572998, -1.0554511547088623, 0.23649974167346954, 0.5690217018127441, -0.980722963809967, -0.5635999441146851, 0.15059900283813477, -1.7529678344726562, -1.0235050916671753, 0.4098169505596161, 0.7666698098182678, -0.1188378855586052, -0.3802543580532074, 0.7382276058197021, -0.10219934582710266, -1.4588295221328735, 0.38699331879615784, 1.0167337656021118, 0.37560969591140747, -0.4240225851535797, -0.40988054871559143, -1.379909634590149, -1.212296724319458, -0.6397968530654907, -0.44181057810783386, -0.7275631427764893, -0.1146225556731224, -0.18242767453193665, 0.19775280356407166, -1.5945684909820557, 0.6383830308914185, 1.8871235847473145, 0.04938897117972374, 0.9252316951751709, 0.12804849445819855, -1.8343377113342285, 1.0518221855163574, -0.18157485127449036, -1.4728692770004272, -1.970973014831543, 0.7056447267532349, 0.5412766933441162, -0.5917161703109741, -0.6857494115829468, 0.7795241475105286, -0.1507100909948349, -0.581726610660553, -1.1112902164459229, -0.16326434910297394, -1.8987568616867065, 1.0078054666519165, -0.08050660043954849, 0.022476330399513245, -1.0632294416427612, -0.10541947185993195, -0.8288461565971375, -0.3717014789581299, -1.3897736072540283, -0.1440010815858841, -0.8150824308395386, -0.4674559235572815, -0.7894020676612854, -1.701206922531128, 1.7105780839920044, -1.7838894128799438, -1.735238790512085, -1.458333969116211, 1.1504348516464233, 0.29045677185058594, -0.7275852560997009, -0.5745667815208435, 1.1067200899124146, -0.2682347297668457, -0.9210349917411804, -0.8774454593658447, 1.508658528327942, -1.4852656126022339, 2.0282328128814697, -0.08611783385276794, 2.414520502090454, 0.003685019677504897, -0.4534202814102173, -1.839913010597229, -2.097137928009033, -0.4597622752189636, -1.0286064147949219, -0.8479698896408081, -0.9331793785095215, -0.6011675596237183, -1.4338034391403198, 1.376326560974121, 0.19944331049919128, 1.2549517154693604, 0.20504367351531982, -0.7525149583816528, 0.9390479326248169, 1.5846469402313232, -0.12671546638011932, 0.5069642066955566, 0.3223477602005005, 0.013150394894182682, 0.9637295007705688, 0.5104351043701172, 1.1516213417053223, -1.003373384475708, -0.7262334823608398, 1.1536915302276611, -0.27149006724357605, 0.4439160227775574, -1.2413203716278076, 0.5821258425712585, 1.3450095653533936, -0.7854627370834351, 0.5004929304122925, 0.294487327337265, -0.08884032070636749, -1.2492597103118896, 0.5223593711853027, 0.692197322845459, 0.09462087601423264, -0.9519886374473572, 0.8945522904396057, 0.3187730610370636, -0.17627626657485962, 0.7983493804931641, -0.5540889501571655, 0.11966636776924133, 0.25988510251045227, 0.41788357496261597, -0.9447782635688782, -0.10008028894662857, 0.10562258958816528, 0.2795087993144989, 0.5295594930648804, -0.12456873804330826, 0.907156765460968, -0.24767181277275085, -0.9085067510604858, 0.31043946743011475, -0.6744693517684937, -0.7227990627288818, 2.059023380279541, 0.11877235770225525, 0.09998034685850143, 1.199330449104309, -0.9525846242904663, -0.928317666053772, 2.0629827976226807, -0.15028266608715057, -0.4416791498661041, 0.8215929865837097, -2.1518454551696777, 0.18966124951839447, -0.3849167227745056, 1.0263193845748901, 0.676559567451477, 0.7722954154014587, 1.1868877410888672, 0.350385844707489, -0.9111401438713074, 0.805947482585907, 0.6542637944221497, 1.2183818817138672, 0.5978481769561768, 0.22918297350406647, 1.7093889713287354, -1.0732231140136719, 0.5985800623893738, -0.8281452655792236, -1.1071218252182007, 0.22741949558258057, -0.1112319603562355, -2.049468994140625, 0.7117647528648376, -0.2629334330558777, 0.7363972067832947, 1.5751018524169922, -0.04913283511996269, -0.7706717848777771, 0.043100204318761826, -1.0265456438064575, -0.460082471370697, 0.45473217964172363, 0.15338993072509766, 0.15189966559410095, -1.0335545539855957, 1.1003397703170776, -3.669750213623047, -0.16193237900733948, -0.34135231375694275, -1.1252572536468506, 0.19284959137439728, -0.6199658513069153, -1.1842275857925415, -0.573529839515686, 0.6224808692932129, 0.6684168577194214, 0.4766373336315155, -0.8192345499992371, -0.5164009928703308, 0.9064750671386719, -0.7741445899009705, 0.4743967354297638, -0.651292085647583, 0.3035581111907959, -0.16629305481910706, -0.5654815435409546, 0.18262335658073425, 1.7622429132461548, 0.5244608521461487, 0.9236297011375427, -0.11464057117700577, 0.46515101194381714, 0.7953803539276123]], "dtype": "float32"}, {"__nd__": [-0.9682217836380005, 0.19628793001174927, -1.523048996925354, -0.23008966445922852, 1.408836007118225, -0.2699860632419586, 0.9608625769615173, -1.7654659748077393, 0.9100448489189148, 0.4476499855518341, -0.40733861923217773, -0.06224893033504486, -1.0732141733169556, 0.02124275080859661, -0.8917216062545776, 0.0703594908118248, 1.9972360134124756, -0.2576278746128082, -0.6300234198570251, -0.16849857568740845, 0.9389374852180481, 0.33887264132499695, -0.2685488760471344, -0.9998002648353577, -0.003219739068299532, -0.26972657442092896, 0.8369897603988647, -1.190698504447937, 1.458655595779419, 0.1987735778093338, 1.559173583984375, -1.3060450553894043, -0.37160569429397583, 2.1443119049072266, 0.656173825263977, -1.2108722925186157, -0.9430818557739258, -0.8698579668998718, -1.826777696609497, 0.8857102990150452, -0.4679156541824341, -1.2735084295272827, 0.24184471368789673, 0.9333181977272034, 0.15015998482704163, -2.051285743713379, -1.6750990152359009, -0.4537631869316101, -0.5712929368019104, -0.3270832896232605, 0.12701241672039032, -0.18756261467933655, 0.14795099198818207, -1.1092661619186401, 0.817699134349823, -0.4549189805984497, -0.3924996852874756, 0.9617794752120972, -0.31095823645591736, -1.5945212841033936, 0.45799365639686584, 1.4858009815216064, 0.14220242202281952, 0.41586169600486755, 0.19454887509346008, 1.6601165533065796, -1.0700312852859497, -0.09465459734201431, 0.9459413886070251, -0.7285336256027222, 1.1551774740219116, 0.7045868635177612, -0.2444351464509964, 0.47342804074287415, 1.093410611152649, -0.6935035586357117, 0.7209507822990417, -0.4883551001548767, -0.4658847749233246, 1.2141715288162231, 0.37224292755126953, -1.473276138305664, -0.41099464893341064, -0.23465916514396667, -0.29051095247268677, -0.8339865207672119, -0.17031089961528778, 1.198685884475708, 0.9459664821624756, -1.3075324296951294, 0.060664255172014236, 2.34619402885437, 0.7493845820426941, 0.7813416123390198, 0.3049216866493225, 0.14138825237751007, 1.1852439641952515, -1.914198875427246, 0.5720455646514893, 0.4455227851867676, -1.5319797992706299, 0.35447949171066284, -1.2837488651275635, 0.10011664777994156, 0.4294303357601166, 1.4702625274658203, -0.8327271938323975, 0.5243749618530273, 0.14417313039302826, -0.5670051574707031, -0.9308947920799255, -1.0623565912246704, -0.018394995480775833, 0.01373283565044403, -0.4154370129108429, 1.220654010772705, 0.23568172752857208, -0.36896565556526184, -0.9809220433235168, 0.6360347270965576, -1.9685086011886597, 0.14454896748065948, 1.7455602884292603, -0.7916691899299622, 0.43445730209350586, -0.5607848763465881, 0.2765791118144989, 0.3429342806339264, 0.3339461088180542, -0.42768386006355286, -0.5796773433685303, 0.06585408002138138, 0.06907516717910767, -0.09394608438014984, 0.3099609911441803, -0.34211382269859314, -2.0886991024017334, -0.18350660800933838, -0.5785431265830994, 0.648973286151886, -3.026671886444092, 0.4467088282108307, 0.28169986605644226, -1.0071070194244385, -1.6323771476745605, 0.1349688470363617, -0.4839325547218323, -2.0599894523620605, -1.5259078741073608, -0.4115743041038513, -0.9935299158096313, 2.5980494022369385, 1.0714237689971924, -0.10216031968593597, 0.9509790539741516, -0.31277933716773987, 2.036590576171875, 0.9830791354179382, -0.16386660933494568, -1.579617977142334, 0.5044492483139038, -2.266108989715576, 0.7145084738731384, -0.9550129175186157, 2.676656723022461, -0.2924359142780304, -0.7616414427757263, 1.660778522491455, 0.12324447929859161, -0.10099181532859802, -1.850407600402832, -0.7199512124061584, 0.30596640706062317, 1.5963811874389648, 0.9580208659172058, 0.23132100701332092, 0.12437800318002701, 0.9328256249427795, 0.35190102458000183, -0.33623433113098145, -1.512549638748169, 0.15178513526916504, -0.7209585905075073, -0.7466086745262146, -0.35195931792259216, -1.1355668306350708, 1.1967631578445435, 0.2944320738315582, -0.08884970843791962, -1.8108552694320679, 0.1002589762210846, 0.4705737233161926, -1.8420852422714233, -1.4880657196044922, -0.33310458064079285, 1.0366215705871582, 2.846524477005005, 1.2024378776550293, -0.7867338061332703, -0.034587178379297256, 2.3555877208709717, 0.6274760365486145, 0.5894920825958252, 0.4643808603286743, -0.3445729911327362, -2.266435384750366, -1.1754472255706787, -0.6883461475372314, 0.5239040851593018, 0.48112231492996216, 0.2195063829421997, -0.44591155648231506, 1.2949306964874268, 0.5906621813774109, 0.19787563383579254, -0.657578706741333, 0.16007666289806366, 0.7468903660774231, 0.8364284038543701, -0.173882856965065, 0.47256597876548767, -0.12019655108451843, 0.7183425426483154, -0.2244916707277298, 2.04794979095459, 1.659180760383606, -1.1018236875534058, 0.2989025413990021, 0.6336806416511536, 0.8996548056602478, -1.4385148286819458, -0.895552396774292, 0.9574432969093323, 0.36857450008392334, 1.5087854862213135, 1.2492430210113525, -0.7843838930130005, 0.10052832216024399, -0.4302726089954376, 0.4315440058708191, 1.8671705722808838, -0.13737592101097107, 1.1648097038269043, 0.47910067439079285, 0.4525938034057617, 0.14413954317569733, 0.08939042687416077, 0.8140241503715515, -0.380504846572876, 0.8416245579719543, 0.205694317817688, -0.7870661020278931, -0.9954274892807007, 1.3433685302734375, -1.4428627490997314, 0.5418674349784851, 3.571053981781006, -1.044961929321289, 0.5073147416114807, -1.3072733879089355, 0.6545960903167725, 0.7363007068634033, 0.3400024175643921, 0.7512924671173096, -0.9706721901893616, 0.1233593076467514, -1.0039942264556885, 0.6747056841850281, -1.395554780960083, 0.5207956433296204, 1.3351619243621826, 1.3605496883392334, 0.1783658266067505, 0.20677664875984192, -0.4998131990432739, 0.17238028347492218, -0.8542757034301758, -1.875617265701294, -0.5556015372276306, 0.6454176902770996, -1.1102746725082397, 0.7031160593032837, 2.1826395988464355, 0.3015354573726654, 0.2493182122707367, -1.989754557609558, -0.38240742683410645, -1.411462664604187, 0.6578568816184998, -0.7718747854232788, -0.045085616409778595, -0.6718062162399292, -0.04434224218130112, -0.21347036957740784, 0.7429747581481934, 2.2763752937316895, 0.44750532507896423, -0.74392169713974, -0.45418864488601685, -0.3478916883468628, -1.2575631141662598, 0.0388471893966198, -0.8301686644554138, -0.1765643060207367, 0.27872300148010254, 1.9397553205490112, -0.08911419659852982, 1.1259739398956299, 1.264604926109314, -1.0478742122650146, -0.09891366213560104, -0.621169924736023, 1.5990407466888428, 0.8292285799980164, 0.5039886832237244, 0.045210741460323334, -1.6761606931686401, -0.3032733201980591, -0.7609503865242004, -0.017931725829839706, -1.2369598150253296, -1.5758055448532104, -1.0761555433273315, 0.6338927745819092, 0.5469971895217896, 0.20471954345703125, 0.16163033246994019, -0.42164355516433716, 0.7057180404663086, -0.7438172698020935, -0.14026014506816864, 0.6851369738578796, 0.746417224407196, -0.9391039609909058, -0.31287336349487305, 0.7026418447494507, 0.13078275322914124, 0.36996957659721375, 0.8617795705795288, 0.28978151082992554, -0.9556334018707275, -1.0219422578811646, 0.1981438845396042, 1.5853767395019531, -0.3466319143772125, 0.9425997734069824, -0.14806696772575378, 1.6918705701828003, -1.8960176706314087, -1.2502294778823853, 1.1423295736312866, 1.1018521785736084, -0.40460285544395447, 0.977310299873352, -0.3690822124481201, -0.09118455648422241, 0.5717279314994812, -0.050267089158296585, -0.35863614082336426, -1.4723223447799683, -1.1925022602081299, -0.766712486743927, 1.5239040851593018, -1.0734562873840332, 0.639585554599762, -1.031470537185669, 0.4206962585449219, -0.0926271304488182, -1.3866442441940308, 0.9716344475746155, -0.6117427349090576, -1.3242015838623047, -0.22971445322036743, 0.08353642374277115, 0.5720811486244202, -1.010250449180603, -1.2197799682617188, 1.4141170978546143, 1.2943909168243408, -0.03271155059337616, 0.5568323135375977, 1.059363842010498, -0.1965726912021637, -0.5558412671089172, 0.12260673195123672, -0.2780805230140686, -0.3365243673324585, 0.9673866629600525, 1.2699335813522339, 0.45518264174461365, 0.873757004737854, -1.068564534187317, 0.5073769092559814, 0.7767857313156128, 0.41016748547554016, -1.5888066291809082, -1.017516851425171, 0.5058633089065552, -1.5818835496902466, 0.27452924847602844, -0.07397495210170746, 1.9576538801193237, -0.6512163877487183, -0.5531241297721863, 1.207229733467102, 1.2279393672943115, -1.0943412780761719, -0.11741086840629578, 0.22873571515083313, -0.2658291459083557, 0.00610074819996953, 0.6125863194465637, -2.2210769653320312, 0.6866125464439392, -0.10325466841459274, 0.19849056005477905, 0.8451637625694275, 0.23874272406101227, -0.25565221905708313, 1.5896520614624023, -0.774238109588623, -1.240680456161499, -2.720247745513916, 0.9895909428596497, -0.9325984716415405, 1.1413458585739136, 0.9283061027526855, 0.9587598443031311, -0.9819620847702026, 0.5024166703224182, 1.5422775745391846, -0.5056378841400146, -0.8807815313339233, 0.1257985234260559, -0.9158782362937927, -0.45789414644241333, -0.04150208458304405, -0.13377778232097626, 0.2602476179599762, -0.8485060334205627, -0.8160910606384277, -0.07020393759012222, 1.1198450326919556, 0.31747299432754517, -2.5135862827301025, 0.025722261518239975, -0.5064250826835632, -0.5280744433403015, -0.24759307503700256, -0.40773600339889526, -0.2654624879360199, 0.670642614364624, 1.25191068649292, -0.5523533821105957, -0.10212256014347076, 1.4616849422454834, -1.1851069927215576, -0.21136125922203064, 0.2623559534549713, -0.6852779984474182, -0.4039058983325958, 1.4794652462005615, 0.581179678440094, -1.3111681938171387, 2.989886522293091, 0.40906962752342224, 0.40357816219329834, -0.4177395701408386, 0.6814571022987366, 0.5225217342376709, -0.0075188144110143185, 0.003909142687916756, 0.04383036121726036, -0.9438523650169373, 0.9070842266082764, -0.6314817070960999, -1.7933300733566284, -2.4515912532806396, 0.13966993987560272, -1.8402873277664185, -0.10751774162054062, 0.15373389422893524, -0.18102286756038666, 1.5273232460021973, -0.44495487213134766, 0.033414263278245926, 0.43510934710502625, 1.2443727254867554, -1.5704257488250732, 2.223480463027954, -0.4430244266986847, 1.6962605714797974, -0.8374854922294617, -0.4949113130569458, 0.4292941391468048, -0.3115409016609192, 0.3569965362548828, -0.28419962525367737, 1.9791998863220215, -0.8388999104499817, 0.3481568396091461, 1.2561229467391968, 0.31246429681777954, -1.824511170387268, 1.2450746297836304, -1.2483761310577393, -0.856722891330719, 1.557724118232727, 1.5108234882354736, -0.14106422662734985, 1.425358533859253, -1.5074087381362915, 0.1499762237071991, -0.6306988596916199, -1.2796698808670044, -2.176607370376587, 1.2516311407089233, -0.06269795447587967, 0.9317043423652649, 0.5640975832939148, -0.1716327667236328, -0.04165904223918915, -1.558136224746704, -0.3023213744163513, 1.0653637647628784, 1.5032827854156494, -0.8554729223251343, 1.6068772077560425, -1.217604160308838, 0.2016545981168747, 0.17897483706474304, 0.9565719366073608, -0.7828795909881592, 0.7943800091743469, -0.5032845139503479, -1.3796319961547852, -0.711090624332428, -0.06304772198200226, 0.9171321392059326, 0.6520865559577942, -0.690142035484314, 0.5110605359077454, 0.7438300251960754, 0.8044586181640625, 0.1875097155570984, -0.6478683352470398, -0.9471654295921326, 0.26350608468055725, 0.6061198115348816, -0.9018678665161133, -0.5949237942695618, 1.3706696033477783, 1.0143108367919922, 0.5095095038414001, 0.1357976496219635, -0.33628299832344055, -0.7915380001068115, 1.0357214212417603, -0.9413321018218994, 1.9271769523620605, 1.6393532752990723, -1.0489933490753174, -1.9148274660110474, 0.0658096969127655, -1.1985315084457397, -0.6017026305198669, 0.27416691184043884, -1.4396237134933472, 0.20636384189128876, 1.265116810798645, 1.2988669872283936, 1.0399802923202515, -0.27114546298980713, -0.4558965265750885, -0.5629481077194214, 0.15104272961616516, 0.733244776725769, -0.10778383165597916, -1.3742773532867432, -1.3115434646606445, -0.7890486121177673, 0.041551798582077026, 0.6205468773841858, -0.09629606455564499, 0.6895302534103394, -0.16489851474761963, 0.19811435043811798, 0.4163074791431427, -0.22510740160942078, 0.8326913118362427, 1.3939552307128906, 0.51848965883255, -0.36924833059310913, 0.10984200239181519, 1.007736325263977, 0.10987567901611328, -0.2583574652671814, -0.5468294620513916, -0.12715119123458862, -0.06453773379325867, -1.6043298244476318, 0.635883092880249, -1.2178367376327515, 0.9079306721687317, -3.1980953216552734, -0.06293895840644836, 1.074455738067627, 1.0003727674484253, -1.5773073434829712, -1.3364975452423096, 0.4367719292640686, -1.2415122985839844, 0.92500239610672, 1.4821839332580566, -1.0475194454193115, -1.193187952041626, -0.36909008026123047, 0.775350034236908, 0.5539817810058594, 0.32491007447242737, -1.4184505939483643, 1.2822731733322144, -0.004669001791626215, 0.6032681465148926, 0.1581808179616928, 0.7904952168464661, 1.2213667631149292, 0.7267633676528931, -2.832127094268799, -0.7860369682312012, 0.6788083910942078, -1.1460514068603516, 2.1107540130615234, -1.0100327730178833, 0.17150017619132996, 0.5429465770721436, 0.8599398732185364, 2.0796525478363037, -1.8895530700683594, 1.0429699420928955, 0.682559609413147, 0.05723238363862038, -0.21801842749118805, -0.7122612595558167, 0.8743491172790527, -0.4154279828071594, 0.6629752516746521, 2.1731534004211426, 1.2998478412628174, 0.8070439696311951, 0.9678642153739929, -3.1575188636779785, 0.8285266160964966, -0.42159101366996765, -1.1335564851760864, 0.8626561164855957, 2.258392333984375, -0.9750540256500244, 0.14135709404945374, -0.7880409955978394, -0.4639289677143097, 1.1810358762741089, -0.4785608649253845, -0.22155210375785828, -0.043245140463113785, -1.5577011108398438, -0.43502968549728394, 0.6750743389129639, 0.7087165713310242, -0.10475058853626251, 1.0604088306427002, 1.3379358053207397, -1.934593677520752, -0.2990269958972931, 0.354604572057724, 0.7629205584526062, 0.0745956301689148, 1.051805853843689, -0.4707045555114746, 0.9780787825584412, -0.8356537818908691, -1.2443586587905884, 1.4831147193908691, 0.30156272649765015, 0.04731608182191849, -1.3913557529449463, 1.0499329566955566, -0.27921566367149353, 0.7576263546943665, -0.3263218104839325, -0.37836745381355286, -0.7730087637901306, -0.6089231371879578, -0.040562212467193604, 0.012084429152309895, -0.03567288815975189, -0.5447304844856262, 0.17509083449840546, 0.5412107110023499, -0.6320661902427673, 0.4126759469509125, 1.2673624753952026, -2.5755834579467773, 0.8240842819213867, -1.3155717849731445, 1.2299833297729492, 0.15436767041683197, 0.9658055305480957, -0.6594353318214417, -0.16436195373535156, -1.6652953624725342, -0.23886266350746155, 0.2134895622730255, 0.6992921233177185, -0.7409974932670593, 0.3323626220226288, -0.5835900902748108, 0.6700927019119263, 0.9494885206222534, -0.3250856101512909, -0.3051423132419586, 0.1514519304037094, -1.9476031064987183, -0.9386926889419556, 0.24180461466312408, -0.4406908452510834, 1.1846871376037598, -1.5240554809570312, 0.8133324980735779, -0.7077491283416748, -1.3096420764923096, 0.24705320596694946, -0.7840072512626648, -1.3627536296844482, 0.4287268817424774, -1.0568461418151855, -1.0923477411270142, 0.29910266399383545, -0.7682100534439087, -0.4047544598579407, 2.0338127613067627, 0.5858068466186523, -0.11890610307455063, -2.5653514862060547, -0.1948525458574295, -0.2546645700931549, 0.3389717638492584, -0.763947069644928, 1.5697752237319946, -0.06773058325052261, -0.33010780811309814, 1.5849488973617554, 0.39728957414627075, 0.09409397095441818, -0.575806200504303, 0.1785467118024826, -0.4021643102169037, -1.0630161762237549, 0.6533995866775513, 0.33675217628479004, 0.5535937547683716, 0.33243417739868164, -0.08814805746078491, -0.8456031680107117, 0.18156698346138, -0.44546565413475037, 1.2578858137130737, 1.7948122024536133, -1.6514002084732056, -0.009800123050808907, -0.4535939395427704, 0.22765588760375977, 1.6694847345352173, 0.8010807633399963, -1.9887088537216187, 1.0644104480743408, -0.5793825387954712, 0.28619062900543213, -0.6842186450958252, -0.7135735750198364, -0.6568562388420105, 0.1355033814907074, 1.5515236854553223, -0.3773248493671417, -0.4239421784877777, -1.1016143560409546, -0.3039343059062958, 0.10249189287424088, -0.6131548285484314, 0.6477965116500854, -0.6090172529220581, -0.11254145950078964, 1.538655161857605, 2.287034511566162, 2.3096017837524414, -0.6230463981628418, -0.3992657959461212, -1.055660367012024, -0.8873588442802429, -0.24583566188812256, 1.0739141702651978, 1.2981420755386353, -0.641448438167572, -0.27590954303741455, 0.969143807888031, 1.1746907234191895, 0.31478792428970337, -0.6586940884590149, 0.3047468662261963, 1.8401837348937988, 1.8427811861038208, 0.009706955403089523, -1.1680347919464111, -0.42394596338272095, -1.592016577720642, 0.09632926434278488, -0.2910239100456238, -0.3594059646129608, 0.5611312985420227, -0.352512001991272, 1.0541774034500122, -0.1189904659986496, 0.20910575985908508, 1.0719432830810547, -1.267754077911377, 1.9192090034484863, -0.14953020215034485, -0.92954421043396, 0.42177218198776245, 2.0541200637817383, -0.4070459306240082, 0.758010983467102, 1.3609693050384521, 1.442633867263794, 0.4251387417316437, -1.083400011062622, -0.3693607449531555, 1.4955600500106812, -1.2167178392410278, 1.2433353662490845, 0.2931414246559143, 0.4938789904117584, 0.22439871728420258, -0.813342273235321, 2.3876795768737793, -1.2903157472610474, -0.14571364223957062, -1.3850232362747192, 0.3586580157279968, 0.5089611411094666, -2.57413387298584, 0.42664045095443726, -0.16117143630981445, 0.8088007569313049, -0.00332450564019382, -1.0225213766098022, 0.0033943962771445513, -0.6833864450454712, 0.7561312913894653, 1.437222957611084, 1.5090620517730713, -0.16209231317043304, -0.9428777098655701, -0.08155035227537155, 1.2673778533935547, 0.33283746242523193, 1.1842281818389893, -0.10741931200027466, 0.013634087517857552, 0.3388095796108246, 0.8829096555709839, 4.170773983001709, 1.6525946855545044, -0.17065125703811646, 0.4026254117488861, -0.5330734252929688, 0.9679514765739441, 0.6339507102966309, -0.22740694880485535, 0.18733105063438416, -0.33369943499565125, -0.3824179470539093, 1.2357474565505981, -0.015054033137857914, -0.7053835988044739, -1.0916975736618042, -0.8016772270202637, -0.9379642605781555, 0.8396048545837402, -0.7221827507019043, 0.22993485629558563, 0.09229160100221634, 0.5342928171157837, -1.1099427938461304, -0.6471750736236572, 0.8144949078559875, 0.6558026075363159, -0.7246293425559998, 1.5047754049301147, 0.28812316060066223, -0.10928593575954437, 0.22712811827659607, -0.9939225912094116, 0.2533779740333557, 0.13082964718341827, 2.5215671062469482, -0.36716440320014954, 0.036504581570625305, -1.6707582473754883, -1.4650660753250122, -0.2033223658800125, 1.9120047092437744, -0.5372013449668884, 0.6291032433509827, 0.02245960384607315, 0.49624118208885193, 1.1009799242019653, -0.21183522045612335, 0.2849160432815552, -0.15281736850738525, -1.943448781967163, 0.4659384787082672, 0.09828895330429077, 0.002775898901745677, 0.8733798861503601, -0.4031735360622406, 1.4535291194915771, -0.06743910163640976, -0.8641383647918701, 0.37992194294929504, 1.9777802228927612, -0.5892318487167358, 0.1805817186832428, -1.6290843486785889, 1.2861993312835693, -1.0620708465576172, -1.0770665407180786, -0.1601102203130722, -0.8072815537452698, -0.15937919914722443, -1.164299488067627, -0.5436157584190369, 0.17869658768177032, 1.0195327997207642, -0.9070866703987122, -1.4976458549499512, -0.5009026527404785, 0.12399870902299881, -0.25134512782096863, 0.2701282799243927, -0.5523824095726013, 0.19421157240867615, 1.122056007385254, -1.0181872844696045, -0.193893700838089, -0.004274149890989065, -0.44976314902305603, -0.6157995462417603, -2.481045722961426, -0.6903272271156311, -0.002242738613858819, 0.10871001332998276, 1.6153068542480469, -1.4054772853851318, -1.7980031967163086, -0.016983425244688988, -0.49840328097343445, 0.4528944194316864, -0.7933023571968079, -1.0992746353149414, -0.1822766810655594, -0.5455421805381775, 0.9068720936775208, -0.6685386300086975, -0.39360311627388, -0.23667818307876587, -1.0692968368530273, -1.8695448637008667, -0.12961551547050476, -0.03454574942588806, -0.8513452410697937, -0.11295592039823532, -1.1676105260849, -0.07714886218309402, -0.7205549478530884, -1.2702691555023193, 0.5374324917793274, 1.254980444908142, 0.011566705070436, 0.5104114413261414, 1.9010194540023804, 0.4329902231693268, 1.6169285774230957, -0.15509025752544403, -0.7910445332527161, 1.862630009651184, -1.5925674438476562, -1.3929924964904785, 0.5603170990943909, 0.20340417325496674], "dtype": "float32"}], "e": {"__nd__": [29.138771057128906, -23.183170318603516, -11.56766128540039], "dtype": "float32"}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(matvec, json.loads(_TESTS), 'cuda', *[0.0001, 1e-05])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")